---
title: "🗓️ Week 05: Non-linear algorithms"
subtitle: "Theme: Supervised Learning"
author: "Dr. Ghita Berrada"
date: 16 February 2026
format: 
  html:
    from: markdown+emoji
    page-layout: full
    toc: true
    toc-depth: 2
self-contained: true
jupyter: python3
engine: jupyter
---


# 🎯 Today's Plan (110 minutes)

**Part 1: Completing Week 4 - COMPAS (40 min)**
- KNN recap
- Formal introduction to cross-validation
- Hyperparameter tuning (KNN + regularized logistic regression)
- Fairness: The impossibility theorem

**BREAK (10 minutes)**

**Part 2: Non-Linear Methods on S&P 500 (60 min)**
- Why non-linearity? Stock market setup
- Support Vector Machines (SVM)
- Decision Trees & Random Forest
- Feature selection (brief)
- Reality check: When ML fails

# ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import make_blobs
import missingno as msno
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_val_score, cross_validate
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,precision_recall_curve)
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import TimeSeriesSplit
import shap
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Part 1: Completing Week 4 - COMPAS Analysis (40 min)

## 1.1 Data Preparation (3 min)

We pre-process the data exactly the same way as we did in Week 4 (filtering observations by race, selection of relevant features/columns, categorical variable encoding, full dataset training/test split, definition of predictors and outcome, standardisation):

In [ ]:
# Load the data
compas = pd.read_csv('data/compas-scores-two-years.csv')
compas_filtered = compas[compas['race'].isin(['African-American', 'Caucasian'])].copy()

# Select features and outcome
feature_cols = ['age', 'sex', 'priors_count', 'c_charge_degree']
target_col = 'two_year_recid'
analysis_cols = feature_cols + [target_col, 'race', 'decile_score', 'score_text']
compas_subset = compas_filtered[analysis_cols]
compas_clean = compas_subset.copy()

# Encode categorical variables
compas_clean['sex_encoded'] = (compas_clean['sex'] == 'Male').astype(int)
compas_clean['charge_felony'] = (compas_clean['c_charge_degree'] == 'F').astype(int)
compas_clean['race_encoded'] = (compas_clean['race'] == 'African-American').astype(int)

# Single stratified split on the full dataset
compas_train, compas_test = train_test_split(
    compas_clean,
    test_size=0.3,
    random_state=42,
    stratify=compas_clean[target_col]
)

# Define predictors and outcome
X_train = compas_train[['age', 'sex_encoded', 'priors_count', 'charge_felony']]
y_train = compas_train[target_col]
X_test = compas_test[['age', 'sex_encoded', 'priors_count', 'charge_felony']]
y_test = compas_test[target_col]
race_test = compas_test['race']

print(f"Training: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {X_train.columns.tolist()}")

In [ ]:
# Scale features for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 1.2 KNN Recap

**K-Nearest Neighbors (KNN)**: A simple, intuitive non-parametric algorithm.

**How it works**:

1. Store all training data
2. For a new point:

   - Find K nearest training examples (using distance metric, usually Euclidean)
   - Take majority vote among those K neighbors
3. Predict the most common class

**Example**: If K=5 and among the 5 nearest neighbors we have 3 "recidivate" and 2 "no recidivate", predict "recidivate".

**Key characteristics**:

- No training phase (lazy learning) - just stores data
- Makes predictions on-demand
- Non-parametric (no assumptions about data distribution)
- Can create non-linear decision boundaries

::: {.callout-note}
### What is the "model" in KNN?

Unlike logistic regression, KNN does **not estimate parameters**.

There is:
- No coefficient vector β
- No optimization problem
- No loss function being minimized during training

Instead:

> The "model" is the entire training dataset.

Prediction is defined as:

1. Compute distances from a new point x₀ to all training points.
2. Identify the K nearest neighbors.
3. Apply a local voting rule (classification) or averaging rule (regression).

Formally, for classification:

$$
\hat{y}(x_0) = \text{majority vote among } \mathcal{N}_K(x_0)
$$

This has important consequences:

• **Model complexity is controlled by K**  
• **Memory usage grows with dataset size**  
• **Prediction time grows with dataset size**  
• No explicit global decision boundary is learned  

KNN is sometimes called a *memory-based* or *instance-based* learner.

It approximates a decision boundary locally rather than globally.

Conceptually:

KNN approximates the conditional probability:

$$
P(Y=1 \mid X=x_0)
$$

by looking at nearby empirical frequencies.

It is a non-parametric estimator of the regression function.

:::

::: {.callout-important}
### Why does scaling matter in KNN?

KNN is based on **distance in feature space**.

By default, `scikit-learn` uses **Euclidean distance**:

$$
d(x_i, x_j) = \sqrt{\sum_{k=1}^{p} (x_{ik} - x_{jk})^2}
$$

If one feature has a larger numerical scale, it dominates this sum.

Example (COMPAS):

- age ranges roughly 18–70  
- priors_count may range 0–30  
- binary features range 0–1  

Without scaling:

Differences in age contribute far more to distance than differences in sex.

This implicitly assigns greater importance to age.

Standardization rescales features to:

Mean = 0  
Variance = 1  

So that distance reflects relative variation rather than units.

### Alternative Distance Metrics

Euclidean is default, but not mandatory.

Other possibilities:

• **Manhattan distance (L1):**  
$$
\sum |x_i - x_j|
$$

• **Minkowski distance:** generalization of L1 and L2 norms (i.e Manhattand and Euclidean distances) 

• **Mahalanobis distance:** accounts for covariance structure  

• **Cosine distance:** common in text applications  

Choice of metric changes the geometry of neighborhoods.

KNN = model defined by:

- K
- Distance metric
- Weighting scheme (uniform vs distance-weighted)
:::



**The critical question**: How do we choose K?

Below is the **full replacement**, written cleanly and completely, with:

* Bias–variance properly structured and vivid
* Explicit link between K and overfitting
* Cross-validation expanded (including CV vs final test gap + temporal CV note)
* Fairness section expanded with COMPAS interpretation
* Nothing removed — only strengthened and clarified

You can paste this directly into your lecture.



## 1.3 Bias–Variance Through the Lens of KNN

In supervised learning, prediction error arises from different sources. A useful conceptual decomposition is:

$$
\mathbb{E}[(Y - \hat{f}(X))^2]=
\text{Bias}^2
+
\text{Variance}
+
\text{Irreducible Noise}
$$

:::callout-note

What does
$\mathbb{E}[(Y - \hat{f}(X))^2]$ mean?


This expression represents the **expected squared prediction error**.

Let’s unpack it:

- $Y$: the true outcome (e.g. whether someone reoffends)
- $\hat{f}(X)$: the model’s prediction given features \(X\)
- $Y - \hat{f}(X)$: the prediction error
- $(Y - \hat{f}(X))^2$: squared error (penalizes large mistakes more heavily)
- $\mathbb{E}[\cdot]$: expectation — the average over all possible data points

So the whole expression means:

> The average squared difference between the true outcome and the model’s prediction, across the population.

In words:

It measures how wrong the model is, on average.

In practice, we do not know the true expectation, so we approximate it using validation or test data.
The bias–variance decomposition explains how this expected error
can be broken into systematic error (bias),
instability across samples (variance),
and irreducible randomness.


:::

Although this decomposition is derived formally in regression settings, the intuition applies equally well to classification.

* **Bias** measures how far the average model prediction is from the true underlying relationship.
* **Variance** measures how much model predictions change if we retrain the model on a different sample drawn from the same population.
* **Irreducible noise** reflects randomness in the data that no model can eliminate.

In KNN, the hyperparameter **K directly controls this tradeoff**.

Think of K as controlling how “stubborn” or “impressionable” the model is.



### 🎓 Student Analogy

Imagine students predicting whether someone will pass an exam.

#### K = 1

The student looks at ONE similar past student.

If that student failed → prediction = fail.
If that student passed → prediction = pass.

The student ignores all other evidence.

* Extremely impressionable
* Highly sensitive to one example
* Overreacts to noise
* Predictions change drastically if we swap a single observation

This corresponds to **low bias, very high variance**.



#### K = 100

The student averages over 100 similar past students.

Individual stories matter less.

The prediction reflects a broad average tendency.

* Very stable
* Much less sensitive to individual observations
* But may ignore important local differences

This corresponds to **higher bias, lower variance**.



### ⚖️ What Happens in COMPAS?

Let’s imagine a 22-year-old male with 0 priors.

#### If K = 1:

Prediction depends entirely on the single closest person in feature space.

If that one person happened to reoffend,
our defendant is predicted high risk.

If that one person did not reoffend,
our defendant is predicted low risk.

A single noisy or unusual case can flip the prediction.

Consequences:

* Training error ≈ 0
* Decision boundary extremely jagged
* Very sensitive to small changes in the training data
* High variance



#### Why Training Error = 0 When K = 1

Each training observation is its own nearest neighbour.

When predicting its own label:

The nearest neighbour is itself.

So it is always classified correctly.

This means:

The model perfectly memorizes the training data.

But memorization is not generalization.

When applied to new individuals (test set), performance can deteriorate sharply.

This is the textbook definition of **overfitting**:

Low training error + high generalization error.



#### If K = 200:

Prediction is based on a very large neighbourhood.

The model averages over many individuals.

Local structure is smoothed out.

Young defendants with no priors may receive similar predictions
to older defendants with multiple priors
because the neighbourhood is too large.

Consequences:

* Lower variance
* Higher bias
* Decision boundary overly smooth
* Risk of underfitting



### The Core Tradeoff

Small K:

* Flexible
* Low bias
* High variance
* Overfits easily

Large K:

* Stable
* High bias
* Low variance
* May underfit

There exists an intermediate K that balances these forces.

We cannot know this optimal K in advance.

That is why we need cross-validation (introduced in the next section).



::: {.callout-tip}

### Thought Experiment: What If We Doubled the Dataset?

Suppose we collected twice as many COMPAS observations.

For very small K:

Neighbourhoods would become more stable.
Predictions would fluctuate less across different samples.

Variance would decrease.

Bias, however, would not automatically change.

This means that with more data,
we can afford slightly smaller K
without incurring as much variance.

Optimal model complexity depends on sample size.
:::


## 1.4 Cross-Validation: A Formal Introduction

### The Problem with Single Train-Test Splits

Up to this point, we have evaluated models using a single train–test split.

We split the dataset into:

* A **training set**, used to fit the model
* A **test set**, used to evaluate performance

This approach is simple and intuitive. However, it has important limitations.

When we evaluate a model using a single train–test split, several problems arise.

#### 1. High Variance of the Performance Estimate

When we say the evaluation has “high variance,” we mean the following:

If we reshuffle the COMPAS dataset and create a new train–test split:

* Different observations will fall into the training set.
* Different observations will fall into the test set.
* The model will be fitted on slightly different data.
* The resulting test score may change noticeably.

Even though the underlying population has not changed.

In other words:

The test performance is not a fixed quantity.
It is a random quantity that depends on the particular random split.

This instability is what we mean by high variance in the evaluation procedure.

We are not only concerned about model variance —
the evaluation itself becomes unstable.

#### 2. Luck Factor

A single test set may be:

* Unusually easy
* Unusually difficult
* Unusually representative
* Or unusually unrepresentative

By chance.

With only one test split, we cannot distinguish between:

* Genuine model quality
* Random sampling luck

#### 3. Data Waste

In a single split:

* The test set is never used for training.
* Potentially useful information is not used to fit the model.

This can be costly, especially in smaller datasets.

#### 4. Unreliable for Hyperparameter Tuning

If we use the test set to select hyperparameters, we contaminate it.

Once the test set influences model selection, it is no longer a neutral audit set.

This leads to overly optimistic performance estimates.

### The Core Question

How can we obtain a more stable and reliable estimate of generalization performance, while preserving the integrity of the final test set?

### K-Fold Cross-Validation: The Solution

Cross-validation is performed **only on the training data**.

The final test set from the original split remains untouched.

The key idea is to rotate which observations act as validation data.

#### Procedure

1. Divide the training data into K folds (typically 5 or 10).
2. For each fold:

   * Train the model on K−1 folds.
   * Evaluate on the remaining fold (validation fold).
3. Repeat this K times.
4. Compute the average validation performance.

In 5-fold CV:

```
Iteration 1: [Valid][Train][Train][Train][Train]
Iteration 2: [Train][Valid][Train][Train][Train]
Iteration 3: [Train][Train][Valid][Train][Train]
Iteration 4: [Train][Train][Train][Valid][Train]
Iteration 5: [Train][Train][Train][Train][Valid]
```

Each observation:

* Is used for training K−1 times.
* Is used for validation exactly once.

The **cross-validation score** is the average of the validation scores.

This provides a more stable estimate of expected out-of-sample performance.

### Stratified K-Fold

In classification problems, we use stratified folds.

This ensures that each fold preserves class proportions.

In recidivism prediction:

If we did not stratify, some folds might contain disproportionately many recidivists or non-recidivists.

This would distort validation performance.

### Example 1: Logistic Regression — Cross-Validation Without Tuning

We now compute:

* Mean training performance (within CV)
* Mean validation performance (CV score)
* Final test performance

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)

cv_results_lr = cross_validate(
    lr,
    X_train_scaled,
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='balanced_accuracy',
    return_train_score=True
)

print("Logistic Regression - 5-Fold CV Results:")
print(f"Train scores: {cv_results_lr['train_score'].round(3)}")
print(f"Validation scores: {cv_results_lr['test_score'].round(3)}")

mean_train_lr = cv_results_lr['train_score'].mean()
mean_cv_lr = cv_results_lr['test_score'].mean()
std_cv_lr = cv_results_lr['test_score'].std()

print(f"\nMean Train (within CV): {mean_train_lr:.3f}")
print(f"Mean CV (Validation): {mean_cv_lr:.3f} (±{std_cv_lr:.3f})")

lr.fit(X_train_scaled, y_train)
final_test_lr = balanced_accuracy_score(y_test, lr.predict(X_test_scaled))

print(f"\nFinal Test Set Performance: {final_test_lr:.3f}")

#### Interpreting these results carefully

We must be precise about terminology.

**Mean Train (within CV)**
This is the average performance on the training folds across the K CV iterations.

This is not the same as training on the entire training set once.

It is the mean performance across the K “inner” training subsets.

**Mean CV (Validation)**
This is the average performance on the held-out fold in each CV iteration.

This is what we call the cross-validation score.

It estimates expected performance on unseen data.

**Final Test Performance**
This is measured on the untouched test set from the original split.

It should only be computed after model selection is complete.

**Interpretation (using our Observed Numbers)**

We obtain:

* Mean Train ≈ 0.666
* Mean CV ≈ 0.664 (± 0.011)
* Final Test ≈ 0.668

We observe:

1. Mean Train and Mean CV are very close.
   This suggests low model variance.
   The model does not dramatically overfit the training folds.

2. The standard deviation across folds is small.
   This indicates stable performance across different validation splits.

3. Final Test performance is extremely close to Mean CV.
   This suggests that cross-validation provided an accurate estimate of generalization performance.

There is no evidence of substantial overfitting.

Logistic regression appears stable in this feature space.

### Training vs CV vs Final Test — Explicit Definitions

To avoid confusion, we define:

* **Train (CV-train)** = Mean training performance across folds
* **CV (validation)** = Mean validation performance across folds
* **Final Test** = Performance on the untouched test set

We examine two gaps.

#### 1️⃣ Train (CV-train) – CV (validation) Gap

If this gap is large:

* The model fits training folds much better than validation folds.
* This indicates high variance.
* The model is likely overfitting.

If both are low:

* The model performs poorly even on training folds.
* This indicates high bias.
* The model is underfitting.

If the gap is small:

* The model is stable.
* Bias–variance balance is reasonable.

#### 2️⃣ CV – Final Test Gap

If final test performance is much lower than CV:

* Hyperparameter tuning may have overfit to CV folds.
* Model selection may be too optimistic.

If final test ≈ CV:

* Cross-validation estimate was reliable.
* Generalization is stable.

Ideally:

Mean Train ≥ Mean CV ≈ Final Test.

### Example 2: Logistic Regression — Hyperparameter Tuning

In [ ]:
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='balanced_accuracy',
    return_train_score=True
)

grid_lr.fit(X_train_scaled, y_train)

print("\nLogistic Regression - Hyperparameter Tuning:")
print(f"Best C: {grid_lr.best_params_['C']}")
print(f"Best Mean CV Score: {grid_lr.best_score_:.3f}")

results_lr = pd.DataFrame(grid_lr.cv_results_)
print(results_lr[['param_C', 'mean_train_score', 'mean_test_score']].round(3))

### Interpretation

Training and validation scores are extremely close across C values:

* There is no large Train–CV gap.
* Increasing C does not substantially increase validation performance.
* Regularization strength has limited influence here.

This suggests that logistic regression is relatively stable in this feature space.

### KNN Hyperparameter Tuning

We now apply the same evaluation logic to KNN.

In [ ]:
param_grid_knn = {'n_neighbors': [1, 3, 5, 7, 10, 15, 20, 30, 40]}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='balanced_accuracy',
    return_train_score=True
)

grid_knn.fit(X_train_scaled, y_train)

print("=" * 60)
print("KNN HYPERPARAMETER TUNING")
print("=" * 60)
print(f"Best K: {grid_knn.best_params_['n_neighbors']}")
print(f"Best Mean CV Score: {grid_knn.best_score_:.3f}")
print("=" * 60)

results_knn = pd.DataFrame(grid_knn.cv_results_)
print(results_knn[['param_n_neighbors',
                    'mean_train_score',
                    'mean_test_score']].round(3))

#### Interpretation of the KNN Results

At small K (e.g., K = 1):

* Mean Train is rather high.
* Mean CV is substantially lower.
* The Train–CV gap is large.

This indicates high variance and overfitting.

The model memorizes training folds but fails to generalize.

As K increases:

* Mean Train decreases.
* Mean CV increases.
* The gap narrows.

Variance decreases.

At K = 40:

* Mean Train and Mean CV are close.
* This suggests a better bias–variance balance.

### Quick note: hyperparameter tuning and the risk of overfitting

We now introduce an important subtlety.

Hyperparameter tuning itself can lead to overfitting.

Why?

Because during tuning:

- We evaluate many hyperparameter combinations.

- We select the one with the highest cross-validation score.

- This selection is based on performance on the validation folds.

Even if each model individually is not overfitting badly,
the act of selecting the best-performing configuration can exploit random fluctuations in validation scores.

This phenomenon is sometimes called **model selection overfitting.**

We may choose a hyperparameter value that performed best partly due to chance variation across folds.

Therefore:

Even after cross-validation-based tuning,
we still require a completely untouched final test set.

The final test set acts as an independent audit of the entire model selection procedure.

Without it, we risk reporting an optimistically biased estimate.

### Visualising the bias/variance tradeoff

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    results_knn['param_n_neighbors'],
    results_knn['mean_train_score'],
    marker='o',
    linewidth=2,
    label='Mean Train (CV-train)'
)

plt.plot(
    results_knn['param_n_neighbors'],
    results_knn['mean_test_score'],
    marker='s',
    linewidth=2,
    label='Mean CV (Validation)'
)

plt.axvline(
    grid_knn.best_params_['n_neighbors'],
    linestyle='--',
    label=f"Optimal K = {grid_knn.best_params_['n_neighbors']}"
)

plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Balanced Accuracy')
plt.title('KNN: Bias–Variance Tradeoff')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Interpreting the KNN Curve

At very small K (e.g., K = 1):

- Mean Train is high.

- Mean CV is substantially lower.

- The Train–CV gap is large.

This is high variance.

The model memorizes training folds and generalizes poorly.

As K increases:

- Mean Train decreases.

- Mean CV increases.

- The gap narrows.

Variance decreases.

At large K (e.g., K = 40):

- Mean Train and Mean CV are close.

- Bias increases.

- Variance decreases.

- Overall validation performance stabilizes.

### Final Test Evaluation for KNN

In [ ]:
best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test_scaled)

final_test_knn = balanced_accuracy_score(y_test, y_pred_knn)

print(f"\nFinal Test Set Performance (K={grid_knn.best_params_['n_neighbors']}):")
print(f"Balanced Accuracy: {final_test_knn:.3f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_knn):.3f}")
print(f"F1 Score: {f1_score(y_test, y_pred_knn):.3f}")

Final test performance ≈ mean CV (balanced accuracy of 0.659 vs 0.664):

* Cross-validation estimate was reliable.
* Model selection did not substantially overfit.

Notably:

KNN and logistic regression achieve very similar validation performance.

Increasing flexibility does not automatically yield better predictive performance.


## 1.5 Fairness: The Impossibility Theorem

**Question**: Does our KNN model exhibit racial bias?

We analyze race to evaluate fairness, not to use it for prediction.

### Base Rates

In [ ]:
test_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_pred_knn,
    'race': race_test
}).reset_index(drop=True)

black = test_df[test_df['race'] == 'African-American']
white = test_df[test_df['race'] == 'Caucasian']

base_rate_black = black['y_true'].mean()
base_rate_white = white['y_true'].mean()

print("BASE RATES (Actual Recidivism):")
print(f"Black: {base_rate_black:.3f} (n={len(black)})")
print(f"White: {base_rate_white:.3f} (n={len(white)})")
print(f"Difference: {abs(base_rate_black - base_rate_white):.3f}")

if abs(black['y_true'].mean() - white['y_true'].mean()) > 0.05:
    print("\n⚠️  Base rates differ → Impossibility theorem applies!")

The base rate is the **actual outcome frequency** in each group.

Here, Black defendants reoffend at a substantially higher observed rate than White defendants in the dataset.

If Black defendants have a higher base rate in the dataset, this means:

$$
P(Y=1∣Race=Black)≠P(Y=1∣Race=White)
$$

At this stage, we do **not** attempt to explain why these base rates differ.
The key point is that **they do differ**, and this fact alone constrains what fairness criteria can be satisfied.

If base rates differ between groups, fairness tradeoffs become unavoidable.


### Three Fairness Definitions

1. Demographic Parity (Statistical Parity)

**Definition:** Probability of positive prediction should be equal across groups.

$$
P(\hat{Y}=1 | A=a) = P(\hat{Y}=1 | A=b)
$$

**Example:** 40% of Black defendants predicted high-risk → 40% of White defendants should also be predicted high-risk.

**Focuses on:** Equality of treatment (same positive prediction rate)

2. Equalized Odds (Error Rate Balance)

**Definition:** True positive rate AND false positive rate should be equal across groups.

$$
P(\hat{Y}=1 | Y=1, A=a) = P(\hat{Y}=1 | Y=1, A=b) \quad \text{[Equal TPR]}
$$
$$
P(\hat{Y}=1 | Y=0, A=a) = P(\hat{Y}=1 | Y=0, A=b) \quad \text{[Equal FPR]}
$$

**Example:** Among those who reoffend, model flags Black and White defendants at same rate. Among those who don't reoffend, false alarm rates are also equal.

**Focuses on:** Equality of error rates

**This is what ProPublica argued COMPAS violated.**

3. Predictive Parity (Calibration)

**Definition:** Among those predicted positive, probability of actually being positive should be equal across groups.

$$
P(Y=1 | \hat{Y}=1, A=a) = P(Y=1 | \hat{Y}=1, A=b)
$$

**Example:** If model predicts high-risk, that prediction should be equally accurate for Black and White defendants.

**Focuses on:** Equality of predictive accuracy

**This is what Northpointe argued COMPAS satisfied.**

::: callout-warning
### Why differing base rates change everything

When groups have different base rates for the outcome:

- Equal positive prediction rates (demographic parity)
- Equal error rates (equalized odds)
- Equal predictive accuracy (predictive parity)

**cannot all hold at once** for a non-trivial classifier.

This is not a modelling failure — it is a mathematical constraint.

(It's called the fairness impossibility theorem!)

Any model that predicts better than random must trade off between these fairness notions.
:::

:::callout-tip
### Why differing base rates force trade-offs

Suppose we have **1,000 defendants** in each group.

#### Step 1: Different base rates (the only assumption)

Assume the observed recidivism rates are:

* **Black defendants:** 50% reoffend → 500 out of 1,000
* **White defendants:** 30% reoffend → 300 out of 1,000

Nothing about modelling yet — this is just the data.

#### Case A: Try to satisfy **predictive parity** (equal accuracy when flagged)

Suppose the model predicts **“high risk”** for:

* 400 Black defendants
* 200 White defendants

And suppose predictive parity holds with **precision = 70%** for both groups.

That means:

**Black defendants**

* 400 flagged as high risk
* 70% actually reoffend → **280 true positives**
* 30% do not → **120 false positives**

**White defendants**

* 200 flagged as high risk
* 70% actually reoffend → **140 true positives**
* 30% do not → **60 false positives**

✅ Predictive parity holds
❌ But now look at error rates:

* Black FPR = 120 / 500 non-reoffenders = **24%**
* White FPR = 60 / 700 non-reoffenders = **8.6%**

➡️ **False positive rates differ sharply**
➡️ **Equalized odds is violated**

#### Case B: Try to satisfy **equalized odds** instead

Now suppose we force:

* False positive rate = 10% for both groups
* True positive rate = 60% for both groups

Let’s compute what that implies.

**Black defendants**

* True positives = 60% of 500 = **300**
* False positives = 10% of 500 = **50**
* Total flagged = **350**

Precision = 300 / 350 ≈ **86%**

**White defendants**

* True positives = 60% of 300 = **180**
* False positives = 10% of 700 = **70**
* Total flagged = **250**

Precision = 180 / 250 = **72%**

✅ Equalized odds holds
❌ Predictive parity fails (precision differs)

#### Case C: Try to satisfy **demographic parity**

Now suppose we force **equal positive prediction rates**:

* 300 Black defendants flagged
* 300 White defendants flagged

Given different base rates, the composition must differ.

Even if the model is doing its best:

* The flagged Black group will contain **more true positives**
* The flagged White group will contain **more false positives**

So either:

* precision differs, or
* error rates differ, or
* both

➡️ Demographic parity cannot coexist with the other criteria.

### The unavoidable conclusion

Once base rates differ:

* **Equal precision** ⇒ unequal error rates
* **Equal error rates** ⇒ unequal precision
* **Equal prediction rates** ⇒ unequal accuracy

There is no way to make the same people:

* be flagged at the same rate,
* make the same kinds of mistakes,
* and have predictions be equally reliable,

**all at once**.

### Why this is not a modelling failure

Nothing above depends on:

* the algorithm,
* the loss function,
* optimisation,
* or bias in training.

The conflict comes from **conditioning on different quantities** when group outcome frequencies differ.

That’s why fairness debates are about **which error you are willing to tolerate**, not about “fixing” the model.


> When groups differ in how often the outcome occurs, you cannot make predictions that are equally frequent, equally wrong, and equally accurate at the same time.

:::


### Demographic parity check


In [ ]:
# Positive prediction rates
ppr_black = black['y_pred'].mean()
ppr_white = white['y_pred'].mean()

print("\nDemographic Parity:")
print(f"Black defendants predicted to reoffend: {ppr_black:.3f}")
print(f"White defendants predicted to reoffend: {ppr_white:.3f}")
print(f"Difference: {ppr_black - ppr_white:.3f}")

if abs(ppr_black - ppr_white) < 0.05:
    print("✓ Approximately satisfies demographic parity")
else:
    print("✗ Violates demographic parity")

The model predicts recidivism for Black defendants at **nearly twice** the rate of White defendants.

This violates demographic parity because the **decision rate itself** differs sharply across groups.

This result largely mirrors the difference in base rates: a model that responds to underlying outcome differences will generally produce unequal positive prediction rates.

### Equalized Odds Check

In [ ]:
def compute_equalized_odds(df):
    y_true = df['y_true']
    y_pred = df['y_pred']
    
    # True Positive Rate (Recall for positive class)
    tpr = recall_score(y_true, y_pred, pos_label=1)
    
    # True Negative Rate (Recall for negative class)
    tnr = recall_score(y_true, y_pred, pos_label=0)
    
    # False Positive Rate = 1 - TNR
    fpr = 1 - tnr
    
    return tpr, fpr

tpr_black, fpr_black = compute_equalized_odds(black)
tpr_white, fpr_white = compute_equalized_odds(white)

print("\nEqualized Odds:")
print("True Positive Rate (Recall):")
print(f"  Black: {tpr_black:.3f}")
print(f"  White: {tpr_white:.3f}")
print(f"  Difference: {tpr_black - tpr_white:.3f}")

print("\nFalse Positive Rate:")
print(f"  Black: {fpr_black:.3f}")
print(f"  White: {fpr_white:.3f}")
print(f"  Difference: {fpr_black - fpr_white:.3f}")

if abs(tpr_black - tpr_white) < 0.05 and abs(fpr_black - fpr_white) < 0.05:
    print("\n✓ Approximately satisfies equalized odds")
else:
    print("\n✗ Violates equalized odds")

Equalized odds fails in two ways:

- Black defendants who reoffend are more likely to be correctly flagged (higher TPR)

- Black defendants who do not reoffend are also more likely to be falsely flagged (higher FPR)

This asymmetry implies a higher burden of **false alarms** for Black defendants, a central concern in the ProPublica critique of COMPAS.

### Predictive Parity Check

In [ ]:
def compute_precision(df):
    return precision_score(df['y_true'], df['y_pred'], pos_label=1)

prec_black = compute_precision(black)
prec_white = compute_precision(white)

print("\nPredictive Parity (Precision):")
print(f"  Black: {prec_black:.3f}")
print(f"  White: {prec_white:.3f}")
print(f"  Difference: {prec_black - prec_white:.3f}")

if abs(prec_black - prec_white) < 0.05:
    print("✓ Approximately satisfies predictive parity")
else:
    print("✗ Violates predictive parity")

Predictive parity is approximately satisfied: when the model predicts recidivism, it is similarly accurate for Black and White defendants.

This comes at the cost of violating both demographic parity and equalized odds.
Achieving equal predictive accuracy requires allowing different error rates when base rates differ.

### Summary Table

In [ ]:
fairness_summary = pd.DataFrame({
    'Metric': ['Base Rate (Actual Recid)', 'Positive Prediction Rate', 
               'True Positive Rate (Recall)', 'False Positive Rate', 
               'Precision (PPV)'],
    'Black Defendants': [base_rate_black, ppr_black, tpr_black, fpr_black, prec_black],
    'White Defendants': [base_rate_white, ppr_white, tpr_white, fpr_white, prec_white],
    'Difference': [
        base_rate_black - base_rate_white,
        ppr_black - ppr_white,
        tpr_black - tpr_white,
        fpr_black - fpr_white,
        prec_black - prec_white
    ]
})

print("\nFairness Metrics Summary:")
print(fairness_summary.round(3).to_string(index=False))

Our model reproduces the classic COMPAS tension:

- Predictive parity holds

- Demographic parity fails

- Equalized odds fails

This is not because the model is “biased” in a narrow technical sense, but because it is operating in a setting where outcome rates differ across groups.

Which fairness definition should matter most is not a statistical question.
It depends on normative judgments about harm, responsibility, and acceptable trade-offs.


::: callout-important
### There is no single “fair” model

Fairness metrics formalise **different ethical priorities**:

- Demographic parity prioritises equal treatment
- Equalized odds prioritises equal error burdens
- Predictive parity prioritises equal reliability of predictions

When base rates differ, improving one typically worsens another.

Fairness in machine learning is therefore about **making trade-offs explicit**, not eliminating them.
:::

#### So… which fairness constraint *should* have been satisfied in the COMPAS case?

This is the crucial point:
👉 **There is no technically “correct” answer.**

The choice is **ethical and institutional**, not statistical.

That said, we *can* reason carefully.

**Fairness in the COMPAS / criminal justice context**

In criminal justice risk assessment, decisions affect:

* liberty (detention, bail),
* punishment severity,
* surveillance intensity.

So errors are **asymmetric** in harm.

*False positives are especially costly*

A false positive means:

* someone is wrongly classified as high risk,
* faces harsher treatment despite not reoffending.

This harm is **direct, individual, and immediate**.

**Strong argument: Equalized odds should have been prioritised**

Many scholars argue that, in this context, **equalized odds** is the most defensible constraint.

Why?

Because it ensures:

* Similar **false positive rates** across groups
  → no group bears a disproportionate burden of wrongful punishment
* Similar **false negative rates**
  → public safety errors are also distributed evenly

In other words:

> People who behave the same way (reoffend or not) should face similar risks of being misclassified, regardless of race.

This directly addresses **procedural fairness**.

**Why predictive parity alone is insufficient**

COMPAS defenders (Northpointe) focused on predictive parity:

> “When COMPAS says someone is high risk, it’s equally accurate across races.”

But this ignores that:

* Black defendants were **far more likely to be labelled high risk**
* Even when they did **not** reoffend

Predictive parity can coexist with **systematically higher false positive rates** — exactly what ProPublica highlighted.

So predictive parity protects **model credibility**, not **individual rights**.

**Why demographic parity is usually inappropriate here**

Demographic parity would require:

* equal numbers of Black and White defendants flagged as high risk

But this would:

* ignore real differences in observed outcomes,
* require deliberate distortion of predictions,
* potentially reduce public safety or inflate false negatives.

Most courts and practitioners reject demographic parity in this setting.

**Conclusion**

> In the context of criminal justice risk assessment, no single fairness definition is universally correct.
> However, given the severe consequences of false positives for individual liberty, many argue that **equalized odds** is the most ethically appropriate constraint.
> The COMPAS controversy illustrates not a failure to meet “fairness”, but a failure to clearly state **which notion of fairness was being prioritised**.

**📌 Takeaway**

> Fairness metrics encode ethical priorities, and when base rates differ, choosing a fairness criterion means choosing which harms are acceptable.

### Practical Implications

**Trade-offs are inevitable:**

* We must choose which fairness metric to prioritize
* This is a **value judgment**, not a technical decision
* Different stakeholders may prefer different definitions

**Context matters:**

* **Criminal justice:** False positives (incorrect high-risk flagging) have severe consequences → equalized odds may be preferred
* **Medical screening:** False negatives (missing disease) may be more harmful → recall is crucial
* **Lending:** Regulatory requirements may mandate demographic parity

**Beyond metrics:**

* **Proxy variables:** Variables correlated with race (e.g., zip code) can perpetuate bias
* **Historical bias:** Training data reflects past discrimination
* **Measurement bias:** What we measure (arrests vs. actual crimes) may be biased
* **Feedback loops:** Predictions influence future outcomes
* **Transparency:** Who can challenge model decisions?



### The Mathematical Impossibility

When base rates differ:

$$
P(Y=1 \mid Race=Black)
\neq
P(Y=1 \mid Race=White)
$$

then it is impossible to simultaneously equalize:

* Prediction rates
* Error rates
* Precision

These impose conflicting probability constraints.

This is not a modeling failure.

It is a structural property of probability.

### Policy Implication

Fairness definitions encode different normative priorities.

* Equalized Odds → equalize error burden.
* Predictive Parity → equalize reliability.
* Demographic Parity → equalize treatment rates.

Choosing among them is not purely technical.

It is a societal decision.

# Part 2: Non-linear methods on stock market data

## 2.1 Why do we need non-linear models?

So far in this course, we have worked mainly with linear regression and logistic regression.

Both belong to the family of **generalized linear models (GLMs)**.

A generalized linear model assumes:

$$
g(\mathbb{E}[Y \mid X]) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p
$$

Let us unpack this carefully.

* $Y$ is the outcome variable.
* $X$ represents our predictors.
* $\mathbb{E}[Y \mid X]$ is the expected value of the outcome given the predictors.
* $g(\cdot)$ is a link function.

In linear regression:

$$
\mathbb{E}[Y \mid X] = \beta_0 + \beta_1 X_1 + \dots
$$

In logistic regression:

$$
\log\left(\frac{P(Y=1)}{1-P(Y=1)}\right)
= \beta_0 + \beta_1 X_1 + \dots
$$

Even though logistic regression transforms probabilities into log-odds, the relationship in the predictors (X) is still **linear and additive**.

This implies:

* Each predictor has a constant marginal effect.
* The effect of moving from (X = 10) to (X = 11) is assumed equal to moving from (X = 30) to (X = 31).
* There are no automatic thresholds.
* Regime changes are not captured unless explicitly modeled.

This is a strong structural assumption.

Financial markets rarely behave in such a smooth proportional wa

## 2.2 Why financial markets are likely non-linear

Financial markets are influenced by:

* investor expectations,
* monetary policy,
* macroeconomic conditions,
* valuation regimes,
* liquidity constraints,
* bubbles and crashes,
* panic and herding behaviour.

Small changes in fundamentals during calm periods may have minimal impact.

The same magnitude change during crisis or speculative periods may trigger dramatic price movements.

This is what we mean by **regime dependence**.

Linear models assume smooth, proportional responses.
Markets often behave in threshold-driven ways.

This motivates non-linear methods.

## 2.3 The S&P 500 dataset

We now use monthly S&P 500 data including:

* Index level
* Dividends
* Earnings
* Consumer Price Index (CPI)
* Long-term interest rate
* Shiller PE10 (CAPE)

In [ ]:
url = "https://raw.githubusercontent.com/datasets/s-and-p-500/master/data/data.csv"
stock = pd.read_csv(url)
stock['Date'] = pd.to_datetime(stock['Date'])

# Restrict to post-1950
stock = stock[stock['Date'] >= '1950-01-01'].copy()

print(f"Date range: {stock['Date'].min()} → {stock['Date'].max()}")
print(f"Observations: {len(stock)}")

The goal here is to predict whether next month's market price will go up or down.

### Lagging explained precisely

Lagging means:

Using only information available at time $t$ to predict time $t+1$.

```python
variable.shift(1)
```

Uses last month’s value.

This prevents data leakage.

#### Why lagging is economically necessary

Imagine:

The Bank of England Monetary Policy Committee meeting.

Members decide on interest rates today.

Do they use:

- Inflation data from next quarter?

- GDP data from next year?

No.

They use:

- Last month’s inflation,

- Last quarter’s GDP,

- Past labour market data.

Policy decisions are always based on past information.

Similarly:

When predicting market returns,
we must only use information known at prediction time.

Using contemporaneous or future values introduces data leakage.

Based on this, many of the variables we'll use as predictors will be lagged.

Now, let's try and understand the data in more details.

### Understanding valuation from first principles

Before constructing predictors, we need to understand what valuation means.

A stock represents ownership in a company.

Owning a stock gives you a claim on:

* Future profits
* Future cash flows
* Future dividends

The price of a stock reflects what investors are willing to pay today for those expected future benefits.


### What does “expensive” mean?

Imagine a small café that earns £10,000 per year in profit.

If someone offers to sell you the café:

- Would you pay £10,000?

- £20,000?

- £200,000?

- £1,000,000?

The answer depends on how much profit the café generates.

If it earns £10,000 per year:

- Paying £20,000 might seem reasonable.

- Paying £1,000,000 would seem absurd.

Why?

Because you would be paying 100 times annual profit.

That would mean it takes 100 years to recover your investment (ignoring growth and discounting).

That’s the core idea behind **valuation**.

### PE ratio explained carefully

$$
PE = \frac{\text{Price}}{\text{Earnings}}
$$

If a company earns £5 per share and its share price is £100:

$$
PE = 100 / 5 = 20
$$

Investors are paying 20 times annual earnings.

Higher PE means more expensive.
Lower PE means cheaper.

### PE10 (Shiller CAPE)

$$
PE10 = \frac{\text{S&P 500 index level}}{\text{Average real earnings over past 10 years}}
$$

Important clarifications:

* The numerator is the entire S&P 500 index.
* The denominator is the 10-year average of inflation-adjusted earnings of the S&P 500 firms.

Why 10 years?

Earnings fluctuate during recessions and booms.
Using a 10-year average smooths cyclical noise.

#### Numerical example

Suppose:

* Index level = 4000
* 10-year average real earnings = 200

Then:

$$
PE10 = 4000 / 200 = 20
$$

If index rises to 6000 but earnings stay at 200:

$$
PE10 = 6000 / 200 = 30
$$

Nothing changed about earnings.
Only price changed.
Valuation increased.

#### Historical PE10 ranges

Rough historical context:

* 10–15 → historically cheap
* 15–20 → historically normal
* 20–30 → expensive
* 30–40 → very expensive
* 40+ → bubble territory

Now consider:

Moving from PE10 = 15 to 16.
Likely minor implication.

Moving from PE10 = 35 to 45.
Potential bubble escalation.

Linear models treat both as identical +1 changes.

Reality likely differs.

#### Making PE10 usable for prediction

When we use PE10 as a predictor, we must respect the information timing constraint.

If we are standing at the end of month $t$ and trying to predict whether the market will go up in month $t+1$, we can only use valuation information that was known at the end of month $t$.

We therefore construct a lagged version:

In [ ]:
stock['PE10_lag1'] = stock['PE10'].shift(1)

This means:

* `PE10_lag1` at month $t$ contains the PE10 value observed at month $t-1$.
* We never use contemporaneous or future valuation information.
* This prevents data leakage.

Economically, this corresponds to the idea that investors form expectations about future returns based on last month’s observed valuation conditions — not on future earnings data.

Even though PE10 is already a long-horizon smoothed variable (10-year average earnings), we still lag it to preserve strict forecasting discipline.

### Dividend yield explained

$$
\text{Dividend Yield} = \frac{\text{Annual dividends}}{\text{Index level}}
$$

Dividend yield measures how much cash income investors receive relative to the price they are paying for the market.

Example:

* Dividends = 80
* Index level = 4000

Dividend yield:

$$
80 / 4000 = 2%
$$

If the index falls to 3000 and dividends remain 80:

$$
80 / 3000 = 2.67%
$$

Nothing changed about the cash flows.
Only the price changed.

The market has become **cheaper relative to income**.



#### Why dividend yield may predict returns

Historically:

* High dividend yields often occur after market declines (bear markets).
* Low dividend yields often occur during strong bull markets.

When dividend yield is high, it often means:

* Prices have fallen relative to cash flows.
* Investors may require higher compensation for risk.
* Future returns may be higher (mean reversion logic).

However:

* During speculative bubbles, yields can remain low for extended periods.
* During crises, yields can spike dramatically.

The relationship is unlikely to be linear across regimes.



#### Constructing a lagged dividend yield

To avoid data leakage, we use only information available at the end of month $t$ to predict month $t+1$.

We therefore construct:

In [ ]:
stock['Dividend_Yield_lag1'] = (
    stock['Dividend'].shift(1) / stock['SP500'].shift(1)
)

This ensures:

* Dividend and price are both taken from month $t-1$.
* The ratio reflects what investors would have observed at that time.
* We never use contemporaneous or future dividend information.

Even though dividends are reported with delay in reality, this lag enforces strict forecasting discipline.



### Earnings yield explained

$$
\text{Earnings Yield} = \frac{\text{Earnings}}{\text{Index level}}
$$

This is the inverse of the price-to-earnings ratio.

If:

$$
PE = 20
$$

Then:

$$
\text{Earnings Yield} = \frac{1}{20} = 5%
$$

This means:

Investors are earning 5% in annual profits relative to the price they pay.

Higher earnings yield implies:

* Lower price relative to profits.
* Cheaper valuation.
* Potentially higher future returns.

Lower earnings yield implies:

* Expensive market.
* Lower expected compensation for risk.



#### Why earnings yield may matter

During distressed markets:

* Prices fall faster than earnings.
* Earnings yield rises sharply.
* This often coincides with bear markets.

During euphoric bull markets:

* Prices rise much faster than earnings.
* Earnings yield falls.
* Valuation becomes stretched.

The predictive relationship may be stronger at extreme values (very high or very low yield).

This again suggests non-linearity.


#### Constructing a lagged earnings yield

In [ ]:
stock['Earnings_Yield_lag1'] = (
    stock['Earnings'].shift(1) / stock['SP500'].shift(1)
)

This uses:

* Earnings observed at month $t-1$
* Price observed at month $t-1$

to forecast month $t+1$.

We respect the same information constraint as with all other features.

### Momentum features explained

#### Return_lag1

In [ ]:
stock['Return_lag1'] = stock['SP500'].pct_change(1).shift(1)

This is the percentage change in the index from two months ago to one month ago.

If index was 4000 and moved to 4200:

Return_lag1 = 5%

This captures short-term momentum.

Empirical finance shows that assets with positive recent returns often continue rising in short horizons.

However, extreme recent gains may signal overheating.

Thus, the effect may not scale linearly.

#### Return_lag3

In [ ]:
stock['Return_lag3'] = stock['SP500'].pct_change(3).shift(1)

Measures cumulative percentage change over past 3 months.

Captures medium-term trend strength.

Strong momentum may reflect:

* Bull market continuation
* Or late-stage speculative behaviour

Again, likely regime-dependent.

### Moving averages explained

In [ ]:
stock['MA3'] = stock['SP500'].shift(1).rolling(3).mean()
stock['MA12'] = stock['SP500'].shift(1).rolling(12).mean()

MA3:
Average index level over last 3 months.

MA12:
Average index level over last 12 months.

If MA3 > MA12:

Recent prices exceed longer-term trend.
Often interpreted as bullish.

If MA3 < MA12:

Recent prices below longer-term trend.
Often interpreted as bearish.

This introduces threshold-type behaviour.
Tree-based models naturally capture such rules.

### Interest rates fully explained

The long interest rate is the yield on 10-year US Treasury bonds.

It plays multiple roles in financial markets:

1. **Borrowing cost**
   Higher rates increase corporate financing costs.

2. **Discount rate for future cash flows**
   Stock prices reflect discounted future earnings.
   A higher discount rate lowers present value.

3. **Safe alternative investment**
   Treasury bonds are considered low-risk assets.
   When bond yields rise, they become more attractive relative to equities.



#### Why the effect may be non-linear

The impact of interest rates depends on their level.

A rise from:

* 1% to 2% (doubling the rate) may be economically large.

A rise from:

* 8% to 9% may have different macro implications.

Additionally:

* Rising rates during strong growth may signal strength.
* Rising rates during fragile conditions may trigger downturns.

Thus, both the level and the change may matter.



#### Constructing lagged interest rate variables

In [ ]:
stock['Interest_Rate_lag1'] = stock['Long Interest Rate'].shift(1)
stock['Interest_Rate_change'] = stock['Long Interest Rate'].diff().shift(1)

This gives us:

* The level of rates last month.
* The change in rates between $t-2$ and $t-1$.

Again, we only use past information.

#### Why include both the level and the change?

These two variables capture **different economic mechanisms**.

**1️⃣ The level of interest rates (`Interest_Rate_lag1`)**

The level reflects the **structural financial environment**.

It tells us:

* Are we in a low-rate regime (e.g. post-2008)?
* Or a high-rate regime (e.g. early 1980s)?

The level affects:

* Long-run discounting of future profits
* The attractiveness of bonds relative to equities
* Corporate financing conditions

In other words:

> The level captures the background macroeconomic regime.

**2️⃣ The change in interest rates (`Interest_Rate_change`)**

The change captures **policy shocks and tightening cycles**.

For example:

* A sudden increase in rates may signal monetary tightening.
* A sharp decrease may reflect crisis intervention.

Markets often react more strongly to *changes* than to levels.

This captures:

* Surprise effects
* Turning points
* Transition between bull and bear markets

In other words:

> The change captures short-run monetary shocks.


#### Why we need both in a predictive model

If we include only the level:

* We miss sudden tightening or easing episodes.

If we include only the change:

* We ignore the broader structural environment.

A rise from 1% to 2% is very different from a rise from 8% to 9%.

The same +1% change has different implications depending on the level.

This interaction between level and change is itself a potential source of **non-linearity**.

Tree-based models and kernel methods can naturally capture such regime interactions without us manually specifying them.


### Inflation

Inflation measures how quickly prices in the economy are rising.

We compute year-over-year inflation:

In [ ]:
stock['Inflation'] = stock['Consumer Price Index'].pct_change(12).shift(1)

This computes:

$$
\frac{CPI_{t-1} - CPI_{t-13}}{CPI_{t-13}}
$$

Why 12 months?

* Monthly inflation is noisy.
* Policymakers focus on annual inflation.
* Investors respond to sustained inflation trends.



#### Why inflation matters for equities

High inflation environments:

* Increase uncertainty.
* Often trigger central bank tightening.
* Raise discount rates.
* Compress equity valuations.

Low and stable inflation environments:

* Associated with monetary stability.
* Often coincide with strong bull markets.

Inflation regimes (e.g., 1970s vs 2010s) create structural shifts in market behaviour.

These structural shifts are unlikely to be well captured by linear models alone.

### Visual evidence of non-linearity

Let us inspect PE10 vs next-month return.

In [ ]:
stock['Forward_return'] = stock['SP500'].pct_change().shift(-1)

plt.figure(figsize=(8,5))
plt.scatter(stock['PE10'], stock['Forward_return'], alpha=0.3)
plt.xlabel("PE10")
plt.ylabel("Next month return")
plt.title("PE10 vs next month return")
plt.axhline(0, linestyle='--')
plt.show()

We observe:

* Weak linear pattern
* Large dispersion
* Extreme negative returns at high PE10 levels

Suggesting non-linear structure.

### Features definition and economic interpretation

| Feature                  | Mathematical definition                                                  | Plain language meaning                                                              | Why it may matter for prediction                                                                                                                                                                                                                                                                                                                                                            |
| ------------------------ | ------------------------------------------------------------------------ | ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Return_lag1**          | $(P_{t-1} - P_{t-2}) / P_{t-2}$                                        | Percentage change in the S&P 500 from two months ago to last month                  | Measures **very short-term momentum**. If the market rose last month, investors may extrapolate the trend (bullish continuation). In contrast, sharp recent declines may signal panic or crisis conditions (bearish sentiment). Short-term momentum can persist in bull markets but often reverses violently in bear markets.                                                               |
| **Return_lag3**          | $(P_{t-1} - P_{t-4}) / P_{t-4}$                                        | Total percentage change over the last three months                                  | Captures **medium-term trend dynamics**. Sustained upward movement over several months is often associated with bull market phases. Sustained downward movement can characterize bear markets. However, during crisis transitions, momentum may break down or reverse non-linearly.                                                                                                         |
| **MA3**                  | $\frac{1}{3} \sum_{i=1}^{3} P_{t-i}$                                   | Average S&P 500 price over the last 3 months                                        | Represents a **smoothed short-term trend**. If the current price is above MA3, markets are trending upward in the short run. Traders often interpret short-term averages as signals of emerging bullish or bearish momentum.                                                                                                                                                                |
| **MA12**                 | $\frac{1}{12} \sum_{i=1}^{12} P_{t-i}$                                 | Average S&P 500 price over the last 12 months                                       | Represents the **longer-term annual trend**. When short-term averages (MA3) rise above long-term averages (MA12), markets are often described as entering a bull phase. When MA3 falls below MA12, this may signal a transition toward bear market conditions.                                                                                                                              |
| **PE10_lag1**            | $\text{Price}_{t-1} / \text{10-year average real earnings up to } t-1$ | Long-run valuation ratio of the S&P 500 using only information available last month | Measures how expensive the overall stock market is relative to long-term earnings fundamentals. Historically, PE10 around 10–15 has coincided with depressed or crisis conditions (bear markets), while values above 30–40 have occurred during speculative or bubble-like bull markets (e.g. late 1990s tech bubble). Very high PE10 may signal overvaluation and increased downside risk. |
| **Dividend_Yield_lag1**  | $\text{Dividend}*{t-1} / \text{Price}*{t-1}$                           | Dividend income relative to market price, using last month’s data                   | Indicates how much cash flow investors receive per unit of price. High dividend yields often occur when prices have fallen (bear markets) or when valuations are low. Lower yields often accompany strong bull markets when prices rise faster than dividend payments.                                                                                                                      |
| **Earnings_Yield_lag1**  | $\text{Earnings}*{t-1} / \text{Price}*{t-1}$                           | Corporate profits relative to market price, based on last month’s information       | The inverse of the price-to-earnings ratio. A high earnings yield suggests that stocks are cheap relative to profits, often occurring in distressed or bearish conditions. Low earnings yield (very high P/E) can signal exuberant bull markets.                                                                                                                                            |
| **Interest_Rate_lag1**   | 10-year Treasury yield at $t-1$                                        | Long-term government bond yield observed last month                                 | Represents the **cost of borrowing** and the **discount rate applied to future corporate profits**. Higher interest rates reduce the present value of future cash flows and provide a safer alternative to equities. Rising rates often pressure stock markets, particularly during late bull market phases.                                                                                |
| **Interest_Rate_change** | $\text{Rate}*{t-1} - \text{Rate}*{t-2}$                                | Change in long-term interest rate over the last month                               | Captures shifts in monetary policy stance. Rapid increases in rates may coincide with tightening cycles that end bull markets. Sudden declines in rates often occur during recessions or financial crises.                                                                                                                                                                                  |
| **Inflation**            | $(\text{CPI}*{t-1} - \text{CPI}*{t-13}) / \text{CPI}_{t-13}$           | Year-over-year inflation rate, based on last month’s data                           | High inflation alters discounting and corporate cost structures. Some bull markets occur in low, stable inflation environments, while high inflation periods (e.g. 1970s) are often associated with volatility and regime shifts. Inflation introduces structural non-linear dynamics.      



> All features are lagged by one period. When predicting month $t+1$, we use only information that would have been available at the end of month $t$. This respects the information structure of financial markets and prevents data leakage.      


### Target construction

In [ ]:
stock['Target'] = (stock['SP500'].shift(-1) > stock['SP500']).astype(int)

Binary classification:

1 = market goes up next month
0 = market goes down

### Looking at missing values

Before constructing the modelling dataset, we must understand **where missing values come from**.

Missing values here are expected for structural reasons:

* `pct_change()` produces missing values at the start of the series
* `rolling()` produces missing values for the first few observations
* `shift(1)` removes the first usable observation
* `shift(-1)` (used for the target) removes the final observation

These are not data errors — they are a consequence of feature engineering.

We inspect missingness **before splitting**.

In [ ]:
# Ensure feature list is defined
features = [
    'Return_lag1',
    'Return_lag3',
    'MA3',
    'MA12',
    'PE10_lag1',
    'Dividend_Yield_lag1',
    'Earnings_Yield_lag1',
    'Interest_Rate_lag1',
    'Interest_Rate_change',
    'Inflation'
]

missing_counts = stock[features + ['Target']].isna().sum()

print("Missing values by column:")
print(missing_counts)

Optional visual inspection:

In [ ]:
msno.matrix(stock[features + ['Target']])

We observe that:

* Early rows missing due to rolling windows and lagging.
* Final row missing `Target` because we shifted forward.



### Assemble modelling dataset (without global row deletion)

We do **not** use:

```python
stock.dropna()
```

That would:

* remove potentially usable observations,
* waste training data,
* contradict best practice regarding missingness.

Instead:

1. Keep only relevant columns.
2. Remove rows where the target is undefined.
3. Leave feature missing values for model-based imputation.

In [ ]:
model_df = stock[['Date'] + features + ['Target']].copy()

# Remove rows where Target is missing
model_df = model_df[model_df['Target'].notna()].copy()

print(f"Observations after removing undefined target: {len(model_df)}")

At this stage:

* Feature missing values remain.
* That is intentional.


### Time-aware split (before imputation)

Because this is time-series data, we must preserve temporal ordering.

We train on earlier years and test on later years.

In [ ]:
split = int(len(model_df) * 0.8)

train = model_df.iloc[:split].copy()
test = model_df.iloc[split:].copy()

X_train = train[features]
y_train = train['Target']

X_test = test[features]
y_test = test['Target']

print(f"Train period: {train['Date'].min()} → {train['Date'].max()}")
print(f"Test period: {test['Date'].min()} → {test['Date'].max()}")

This mimics real forecasting:

* We pretend we are standing in the past.
* We train on historical information.
* We evaluate on genuinely unseen future data.

This prevents look-ahead bias.

### Imputation using bagged trees (non-linear, multivariate)

We now handle missing feature values.

Instead of:

* mean imputation (shrinks variance),
* median imputation (can distort skewed distributions),
* KNN imputation (requires scaling and assumes local similarity),

we use **tree-based iterative imputation**.

Why?

Financial variables often:

* interact non-linearly,
* behave differently across regimes,
* exhibit skewness and outliers.

Tree-based models handle these naturally.

We use a Random Forest inside an iterative imputer.

This effectively learns:

> For each feature with missing values, predict it using all other features via a non-linear ensemble model.

This is conceptually similar to bagged-tree imputation.

We are already using trees before formally introducing them — just for imputation (instead of classification later).

In [ ]:
rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=50,
        random_state=42
    ),
    max_iter=10,
    random_state=42
)

# Fit on training data only
X_train_imputed = rf_imputer.fit_transform(X_train)

# Apply same transformation to test data
X_test_imputed = rf_imputer.transform(X_test)

Important:

* The imputer is fitted **only on training data**.
* The test set is never used during fitting.
* This avoids information leakage.


### Baselines: what are we actually competing against?

Before fitting complex non-linear models, we need to define what “doing nothing intelligent” looks like.

In financial prediction, baselines are not arbitrary numbers — they correspond to **concrete decision strategies**.

#### Baseline 1: Random guessing

In [ ]:
baseline_random = 0.5

This corresponds to:

> Each month, flip a fair coin.
> Heads → predict “Up”
> Tails → predict “Down”

Because this is a binary problem and classes are roughly balanced, expected balanced accuracy = 0.5.

This baseline answers:

> Does our model extract *any* signal at all from the data?

If a model does not beat 0.5 balanced accuracy, it has learned nothing.


#### Baseline 2: Always predict the majority class

In [ ]:
baseline_majority = y_train.value_counts(normalize=True).max()

This corresponds to:

> Always predict the most common outcome observed in the training data.

For example, if historically:

* 55% of months were Up
* 45% were Down

Then this strategy predicts **Up every month**.

This yields 55% accuracy without using any predictors.

This baseline captures the market’s **long-run upward drift**.


### Compute and display baselines

In [ ]:
print(f"Random baseline (coin flip): {baseline_random:.3f}")
print(f"Majority baseline (always predict most frequent class): {baseline_majority:.3f}")

### Why both baselines matter

* Random baseline → tests whether there is *any predictive structure*
* Majority baseline → tests whether the model improves on unconditional frequency

In finance, beating the majority baseline is much harder than beating random.

Any serious model must outperform both.




## 2.4 Support vector machines

Support Vector Machines approach classification differently from logistic regression.

Logistic regression models probabilities:

$$
\log\left(\frac{P(Y=1)}{1-P(Y=1)}\right)
= \beta_0 + \beta_1 X_1 + \dots
$$

SVM does **not** model probabilities.

Instead, it asks:

> What separating boundary maximizes the geometric distance between classes?

This is a geometric optimisation problem.


### 2.4.1 The separating hyperplane defined properly

Let each observation be a vector:

$$
x = (x_1, x_2, \dots, x_p)
$$

An SVM seeks a hyperplane:

$$
w^\top x + b = 0
$$

We define every object carefully.

#### What is $w$?

* $w = (w_1, \dots, w_p)$
* A vector of coefficients
* Determines orientation (tilt) of the hyperplane

### What is $w^\top x$?

$$
w^\top x = w_1 x_1 + w_2 x_2 + \dots + w_p x_p
$$

It is simply a weighted sum of predictors.

### What is $b$?

* Intercept term
* Shifts boundary

### Classification rule

If:

$$
w^\top x + b > 0
$$

→ classify as class +1

If:

$$
w^\top x + b < 0
$$

→ classify as class −1



### 2.4.2 What is the margin?

Among all separating hyperplanes, SVM chooses the one that maximizes the **margin**.

The margin is the perpendicular distance from the decision boundary to the closest points of each class.

Those closest points are called **support vectors**.

Formally:

Decision boundary:

$$
w^\top x + b = 0
$$

Margin boundaries:

$$
w^\top x + b = +1
$$

$$
w^\top x + b = -1
$$

The distance between these two parallel hyperplanes equals:

$$
\frac{2}{|w|}
$$

Where:

$$
|w| = \sqrt{w_1^2 + \dots + w_p^2}
$$



#### From geometry to optimisation

So far, we have described the margin **geometrically**.

But we now need to connect this geometry to an optimisation problem.

The key question is:

> Can we always perfectly separate the two classes?

There are two fundamentally different situations.


**Case 1: Perfectly separable data**

If the data are perfectly separable, then there exist many hyperplanes that classify all observations correctly.

Among all such separating hyperplanes, SVM chooses the one that maximises the margin.

Because:

$$
\text{Margin width} = \frac{2}{|w|}
$$

Maximising the margin is equivalent to **minimising $|w|$**.

This is why, in the perfectly separable case, the SVM optimisation problem can be expressed (informally) as:

> Minimise $|w|^2$
> subject to all points being correctly classified and lying outside the margin.

This is called the **hard-margin SVM**.

In this setting:

* No observation is allowed inside the margin.
* No misclassification is allowed.
* The constraints must be strictly satisfied.

Hard-margin SVM assumes perfect separation is possible.

**Case 2: Overlapping or noisy data**

In most real datasets:

* Classes overlap.
* Noise is present.
* Perfect separation is impossible.

If we insisted on zero classification errors, the optimisation problem would have **no feasible solution**.

So we relax the constraints.

We still:

> Maximise the margin.

But we now allow:

* Some observations to lie inside the margin.
* Some observations to even be misclassified.

However, we penalise those violations.

This gives us the **soft-margin SVM**.

The strength of the penalty is controlled by the parameter:

$$
C
$$

So we now have two distinct but related optimisation settings:

| Setting     | Assumption                  | What is allowed?     | Margin objective                   |
| -- |  | -- | - |
| Hard margin | Perfect separation possible | No violations        | Maximise margin                    |
| Soft margin | Overlap exists              | Violations penalised | Maximise margin subject to penalty |

Importantly:

> **Both hard and soft margin SVMs maximise the margin.**

The difference lies only in whether violations are forbidden (hard) or penalised (soft).



### Why $|w|^2$ appears in the optimisation problem

Because margin width equals:

$$
\frac{2}{|w|}
$$

Maximising the margin
$\Longleftrightarrow$ Minimising  $|w|$

For mathematical convenience (differentiability and convexity), the optimisation problem uses:

$$
|w|^2
$$

instead of $|w|$.

That is why the SVM objective contains a quadratic penalty term in $w$.

We are not modelling probabilities.
We are controlling the geometry of the separating surface.



### Geometric illustration using Plotly

This example constructs two artificial clusters and shows:

* Decision boundary
* Margin lines
* Support vectors

This example constructs two artificial clusters and shows:

* Decision boundary
* Margin lines
* Support vectors

In [ ]:
# ----------------------------------------------------------
# 1. Create synthetic linearly separable 2D data
# ----------------------------------------------------------

X, y = make_blobs(
    n_samples=40,
    centers=2,
    cluster_std=1.2,
    random_state=42
)

# Convert labels from {0,1} to {-1,+1}
y = np.where(y == 0, -1, 1)

# ----------------------------------------------------------
# 2. Fit linear SVM (hard-margin approximation)
# ----------------------------------------------------------

clf = SVC(kernel="linear", C=1e6)
clf.fit(X, y)

w = clf.coef_[0]
b = clf.intercept_[0]

# ----------------------------------------------------------
# 3. Compute decision boundary and margin lines
# ----------------------------------------------------------

x_vals = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300)

# Decision boundary: w1*x1 + w2*x2 + b = 0
decision_boundary = -(w[0]*x_vals + b) / w[1]

# Margin lines: wᵀx + b = ±1
margin_plus = -(w[0]*x_vals + b - 1) / w[1]
margin_minus = -(w[0]*x_vals + b + 1) / w[1]

# ----------------------------------------------------------
# 4. Compute exact perpendicular margin segment
# ----------------------------------------------------------

w_norm = np.linalg.norm(w)
margin_width = 2 / w_norm

# Take ONE support vector that lies on +1 margin
sv = clf.support_vectors_[0]

# Ensure it is actually on +1 margin (numerical precision)
# If not, use one that satisfies the condition
for candidate in clf.support_vectors_:
    if abs(np.dot(w, candidate) + b - 1) < 1e-3:
        sv = candidate
        break

# Unit normal direction
n_hat = w / w_norm

# Move from +1 margin to -1 margin
x_plus = sv
x_minus = sv - (2 / w_norm) * n_hat

# ----------------------------------------------------------
# Alternative candidate hyperplanes (rotated weight vectors)
# ----------------------------------------------------------

def rotate_vector(v, theta):
    R = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])
    return R @ v

thetas = [-0.6, -0.3, 0.3]
candidate_lines = []

for theta in thetas:
    w_rot = rotate_vector(w, theta)
    
    # Keep same midpoint constraint by recomputing intercept
    # Pass through same boundary midpoint x0
    x0 = -b * w / (np.linalg.norm(w)**2)
    b_rot = -np.dot(w_rot, x0)
    
    y_vals = -(w_rot[0]*x_vals + b_rot) / w_rot[1]
    candidate_lines.append(y_vals)



# ----------------------------------------------------------
# 5. Plot with Plotly
# ----------------------------------------------------------

fig = go.Figure()

# Class -1
fig.add_trace(go.Scatter(
    x=X[y==-1][:,0],
    y=X[y==-1][:,1],
    mode="markers",
    marker=dict(size=9, color="royalblue"),
    name="Class -1"
))

# Class +1
fig.add_trace(go.Scatter(
    x=X[y==1][:,0],
    y=X[y==1][:,1],
    mode="markers",
    marker=dict(size=9, color="tomato"),
    name="Class +1"
))

# Support vectors
fig.add_trace(go.Scatter(
    x=clf.support_vectors_[:,0],
    y=clf.support_vectors_[:,1],
    mode="markers",
    marker=dict(size=16, symbol="circle-open", line=dict(width=3, color="green")),
    name="Support vectors"
))

# Decision boundary
fig.add_trace(go.Scatter(
    x=x_vals,
    y=decision_boundary,
    mode="lines",
    line=dict(width=4, color="black"),
    name="Decision boundary (wᵀx + b = 0)"
))

# Margin lines
fig.add_trace(go.Scatter(
    x=x_vals,
    y=margin_plus,
    mode="lines",
    line=dict(dash="dash", width=3, color="black"),
    name="Margin (wᵀx + b = +1)"
))

fig.add_trace(go.Scatter(
    x=x_vals,
    y=margin_minus,
    mode="lines",
    line=dict(dash="dash", width=3, color="black"),
    name="Margin (wᵀx + b = -1)"
))

for i, y_vals in enumerate(candidate_lines):
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines",
        line=dict(dash="dot", width=2),
        name=f"Candidate hyperplane (rotation {thetas[i]:+.1f} rad)"
    ))
# Perpendicular margin width segment
fig.add_trace(go.Scatter(
    x=[x_minus[0], x_plus[0]],
    y=[x_minus[1], x_plus[1]],
    mode="lines",
    line=dict(width=6, color="purple"),
    name="Margin width = 2 / ||w||"
))

# Annotation
fig.add_annotation(
    x=(x_minus[0] + x_plus[0]) / 2,
    y=(x_minus[1] + x_plus[1]) / 2,
    text=f"Margin width = {margin_width:.3f}",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40
)

# ----------------------------------------------------------
# 6. IMPORTANT: Equal axis scaling (fixes geometry visually)
# ----------------------------------------------------------

fig.update_layout(
    title="Linear SVM: Maximum margin geometry",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    width=900,
    height=700,
    yaxis=dict(scaleanchor="x", scaleratio=1)
)

fig.show()


#### What we observe

1. Many hyperplanes could separate the clusters.
2. SVM chooses the one with largest geometric gap.
3. Only the support vectors determine the boundary.
4. Moving other interior points does not change the solution.

This is very different from logistic regression, where all observations influence the likelihood.


:::callout-note
### What is a support vector?

A **support vector** is a training observation that lies on (or inside, in the soft-margin case) the margin boundary and directly determines the position of the separating hyperplane.

In the hard-margin case, support vectors satisfy:

$$
w^\top x_i + b = \pm 1
$$

They are the points **closest to the decision boundary**.

In the soft-margin case, support vectors include:

* Points exactly on the margin
* Points inside the margin
* Points that are misclassified

Only these observations determine the optimal hyperplane.

If you move a non-support point slightly, the boundary does not change.

If you move a support vector, the boundary shifts.

This is why:

> The SVM solution depends only on the support vectors — not on the rest of the dataset.

### Geometric intuition

The SVM maximizes the margin width:

$$
\text{Margin} = \frac{2}{|w|}
$$

The support vectors are exactly the points that “touch” the margin and therefore constrain how wide it can be.

:::


### Soft margin and the role of C

When we move from hard-margin to soft-margin SVM, we introduce a penalty for margin violations.

The parameter $C$ controls the trade-off:

* Large $C$ → heavily penalise violations → narrower margin → lower bias, higher variance
* Small $C$ → tolerate violations → wider margin → higher bias, lower variance

So $C$ plays a role analogous to a regularisation parameter in regression.

It determines how strictly we enforce separation versus how wide we allow the margin to be.


In [ ]:
# ----------------------------------------------------------
# 1. Overlapping synthetic data
# ----------------------------------------------------------

X, y = make_blobs(
    n_samples=60,
    centers=2,
    cluster_std=2.2,
    random_state=42
)

y = np.where(y == 0, -1, 1)

# ----------------------------------------------------------
# 2. Soft-margin SVM
# ----------------------------------------------------------

C_value = 1.0
clf = SVC(kernel="linear", C=C_value)
clf.fit(X, y)

w = clf.coef_[0]
b = clf.intercept_[0]

# ----------------------------------------------------------
# 3. Decision boundary and margins
# ----------------------------------------------------------

x_vals = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 400)

decision_boundary = -(w[0]*x_vals + b) / w[1]
margin_plus = -(w[0]*x_vals + b - 1) / w[1]
margin_minus = -(w[0]*x_vals + b + 1) / w[1]

# ----------------------------------------------------------
# 4. Margin violations
# ----------------------------------------------------------

decision_values = y * (X @ w + b)
violations = decision_values < 1

# ----------------------------------------------------------
# 5. True perpendicular margin segment
# ----------------------------------------------------------

w_norm = np.linalg.norm(w)
margin_width = 2 / w_norm

n_hat = w / w_norm
x0 = -b * w / (w_norm**2)

half_margin = 1 / w_norm
x_plus = x0 + half_margin * n_hat
x_minus = x0 - half_margin * n_hat

# ----------------------------------------------------------
# 6. Plot
# ----------------------------------------------------------

fig = go.Figure()

# ----------------------------------------------------------
# SHADED MARGIN REGION (this is the key improvement)
# ----------------------------------------------------------

fig.add_trace(go.Scatter(
    x=np.concatenate([x_vals, x_vals[::-1]]),
    y=np.concatenate([margin_plus, margin_minus[::-1]]),
    fill='toself',
    fillcolor='rgba(200, 200, 200, 0.25)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    name="Margin band"
))

# ----------------------------------------------------------
# Data points
# ----------------------------------------------------------

fig.add_trace(go.Scatter(
    x=X[y==-1][:,0],
    y=X[y==-1][:,1],
    mode="markers",
    marker=dict(size=8, color="royalblue"),
    name="Class -1"
))

fig.add_trace(go.Scatter(
    x=X[y==1][:,0],
    y=X[y==1][:,1],
    mode="markers",
    marker=dict(size=8, color="tomato"),
    name="Class +1"
))

# Violations (big + bright)
fig.add_trace(go.Scatter(
    x=X[violations][:,0],
    y=X[violations][:,1],
    mode="markers",
    marker=dict(
        size=18,
        color="yellow",
        symbol="x",
        line=dict(width=3, color="black")
    ),
    name="Margin violations (ξᵢ > 0)"
))

# Support vectors
fig.add_trace(go.Scatter(
    x=clf.support_vectors_[:,0],
    y=clf.support_vectors_[:,1],
    mode="markers",
    marker=dict(
        size=16,
        symbol="circle-open",
        line=dict(width=3, color="green")
    ),
    name="Support vectors"
))

# Decision boundary
fig.add_trace(go.Scatter(
    x=x_vals,
    y=decision_boundary,
    mode="lines",
    line=dict(width=4, color="black"),
    name="Decision boundary"
))

# Margin lines (stronger contrast)
fig.add_trace(go.Scatter(
    x=x_vals,
    y=margin_plus,
    mode="lines",
    line=dict(dash="dash", width=3, color="black"),
    name="Margin +1"
))

fig.add_trace(go.Scatter(
    x=x_vals,
    y=margin_minus,
    mode="lines",
    line=dict(dash="dash", width=3, color="black"),
    name="Margin -1"
))

# Perpendicular margin width segment
fig.add_trace(go.Scatter(
    x=[x_minus[0], x_plus[0]],
    y=[x_minus[1], x_plus[1]],
    mode="lines",
    line=dict(width=7, color="purple"),
    name="Margin width = 2 / ||w||"
))

fig.add_annotation(
    x=(x_minus[0] + x_plus[0]) / 2,
    y=(x_minus[1] + x_plus[1]) / 2,
    text=f"Soft margin width = {margin_width:.3f}",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40
)

# ----------------------------------------------------------
# Zoom and equal scaling
# ----------------------------------------------------------

x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

pad_x = (x_max - x_min) * 0.15
pad_y = (y_max - y_min) * 0.15

fig.update_layout(
    title=f"Soft-Margin SVM (C = {C_value})",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    width=850,
    height=700,
    xaxis=dict(range=[x_min - pad_x, x_max + pad_x]),
    yaxis=dict(
        range=[y_min - pad_y, y_max + pad_y],
        scaleanchor="x",
        scaleratio=1
    )
)

fig.show()

## 2.5 SVM on stock market data

We now apply Support Vector Machines to our S&P 500 prediction problem.

Recall:

* Target: 1 if market goes up next month, 0 otherwise.
* Features: lagged valuation, momentum, macro variables.
* Time-aware split: train on earlier years, test on later years.
* Missing values: imputed using bagged trees on training data only.

Because SVMs are sensitive to feature scale, we must **standardise features**.

However, before fitting any model, we must answer an important question:

> Is the dataset class-imbalanced?

This matters because:

* Accuracy becomes misleading under imbalance.
* Balanced accuracy, F1, PR-AUC become more appropriate.
* Majority baseline becomes stronger when imbalance increases.


### 2.5.1 Inspecting class imbalance

#### Distribution in training and test sets

In [ ]:
print("Training set class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest set class distribution:")
print(y_test.value_counts(normalize=True))

The classes are moderately imbalanced in both training and test sets.

Recall earlier baseline:

Majority baseline = 0.610

This already tells us:

> Roughly 61% of months in training were “Up”.

This is economically meaningful:

* Equity markets have a long-run upward drift.
* Always predicting “Up” is not trivial — it encodes historical bias.

#### Imbalance over time

Markets evolve.

Class imbalance may not be stable across decades.

We inspect rolling proportions:

In [ ]:
train_temp = train.copy()
train_temp['Target'] = y_train.values

rolling_up = (
    train_temp
    .set_index('Date')['Target']
    .rolling(window=60)  # 5-year rolling window
    .mean()
)

plt.figure(figsize=(10,5))
plt.plot(rolling_up)
plt.axhline(0.5, linestyle='--', color='black')
plt.title("Rolling 5-year proportion of Up months (training period)")
plt.ylabel("Proportion Up")
plt.xlabel("Date")
plt.show()

**Interpretation of rolling 5-year proportion of Up months**

What the plot shows:

* The dashed line at 0.5 represents equal Up/Down probability.
* The rolling 5-year proportion of Up months fluctuates between roughly **0.47 and 0.75**.
* There are long periods where the market is strongly upward biased (e.g. late 1990s).
* There are periods where the proportion falls close to or even below 0.5 (e.g. mid-1970s, early 2000s).

*Key implications*

1. The class distribution is **not stationary over time**.

2. The majority class changes in strength across regimes.

3. During strong bull regimes (e.g. ~70–75% Up months):

   * A naive “always Up” classifier performs very well.
   * Balanced accuracy may still be 0.5, but raw accuracy and F1 will look high.

4. During weak or crisis regimes:

   * Majority rule becomes much weaker.
   * Predictive models must detect regime shifts to outperform.

5. This plot explains why:

   * F1 can be high even when balanced accuracy is 0.5.
   * Majority baseline is economically meaningful in equity data.
   * Evaluation must be time-aware.

*Conceptual takeaway*

The rolling plot demonstrates:

> Class imbalance in financial returns is regime-dependent, not fixed.

This means:

* Performance must be interpreted in historical context.
* A model that merely captures unconditional upward drift is not learning predictive structure.
* True skill requires detecting when the conditional probability deviates from the long-run average.

### 2.5.2 Scaling (required for SVM)

SVM relies on dot products:

$$
w^\top x
$$

If one feature has scale 1000 and another has scale 0.02,
the larger-scale feature dominates the geometry.

We therefore scale:

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

### 2.5.3 Baseline metrics under imbalance

We now compute baseline performance using **imbalance-appropriate metrics**.

#### Majority classifier predictions

In [ ]:
majority_class = y_train.value_counts().idxmax()

y_pred_majority = np.full_like(y_test, fill_value=majority_class)

print("Majority baseline metrics:")
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_majority), 3))
print("F1 score:",
      round(f1_score(y_test, y_pred_majority), 3))

print("PR-AUC:",
      round(average_precision_score(y_test,
            np.full_like(y_test, fill_value=1, dtype=float)), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_majority))

Important:

* Balanced accuracy equals **0.5** for majority rule because one class is predicted perfectly and the other not at all.
* F1 may look artificially high if the positive class dominates.
* PR-AUC for majority prediction equals the **positive class prevalence**, because the classifier produces a constant score.

This highlights something subtle but critical:

> A model can look strong on F1 while completely failing on the minority class.

#### Random baseline (probabilistic)

We compute metrics consistently using the same framework.

In [ ]:
rng = np.random.default_rng(42)
y_pred_random = rng.choice([0, 1], size=len(y_test))
y_score_random = rng.uniform(0, 1, size=len(y_test))

print("Random baseline metrics:")
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_random), 3))
print("F1 score:",
      round(f1_score(y_test, y_pred_random), 3))
print("PR-AUC:",
      round(average_precision_score(y_test, y_score_random), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_random))

Interpretation:

- Balanced accuracy = 0.571 exceeds 0.5, but this reflects realised sampling variation, not predictive structure.

- The confusion matrix shows errors in both classes consistent with random guessing.

- F1 = 0.643 appears moderate because the positive class (Up months) is frequent.

- PR-AUC = 0.649 is close to the positive class prevalence, as expected under randomness.

This is critical:

> A random classifier can appear “above 0.5” on balanced accuracy and achieve non-trivial F1 when the positive class dominates.

Therefore:

Baseline performance must be interpreted relative to theoretical expectations, not raw magnitudes.

True predictive skill requires performance that systematically exceeds what randomness can produce.

### 2.5.4 Linear SVM

We start with a linear kernel.

In [ ]:
svm_linear = SVC(kernel="linear", C=1.0, random_state=42)
svm_linear.fit(X_train_scaled, y_train)

y_pred_svm_linear = svm_linear.predict(X_test_scaled)
y_score_svm_linear = svm_linear.decision_function(X_test_scaled)

#### Metrics (imbalance-aware)

In [ ]:
print("Linear SVM metrics:")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_svm_linear), 3))

print("F1 score:",
      round(f1_score(y_test, y_pred_svm_linear), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_score_svm_linear), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_svm_linear))

#### Precision–Recall curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_score_svm_linear)

plt.figure(figsize=(6,5))
plt.plot(recall, precision)
plt.axhline(positive_rate, linestyle='--', color='black')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve (Linear SVM)")
plt.show()

Interpretation:

* If the curve lies only slightly above the horizontal baseline → the model extracts only weak ranking signal.
* If the curve closely hugs the baseline → no meaningful improvement over random.
* Even if hard predictions collapse to majority class, a PR curve above baseline suggests some weak ordering information in the decision function.


#### Interpreting performance relative to baselines

We now compare:

| Metric            | Random               | Majority        | Linear SVM      |
| ----------------- | -------------------- | --------------- | --------------- |
| Balanced accuracy | 0.571                | 0.50            | 0.50            |
| F1 (Up class)     | 0.643 | 0.816           | 0.816           |
| PR-AUC            | 0.649       | ≈positive rate  | 0.727           |
| Confusion matrix  | Mixed                | Predicts all Up | Predicts all Up |

Interpretation structure:

* Balanced accuracy = 0.5 → no discrimination between Up and Down.
* Identical confusion matrix to majority → model learned only unconditional upward drift.
* F1 identical to majority → no improvement in minority detection.
* PR-AUC slightly above baseline → possible weak ranking signal, but not enough to alter classification boundary.

In equity prediction, even +0.02 balanced accuracy improvement can be economically meaningful.

Here, we do **not** observe such improvement.

### 2.5.5 Non-linear SVM (RBF kernel)

Markets likely exhibit non-linear structure.

We now use the RBF kernel:

$$
K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)
$$

This allows curved decision boundaries.


:::callout-important

### Why does the SVM model still look linear with a non-linear kernel?

Earlier, we wrote the SVM decision function as:

$$
f(x) = w^\top x + b
$$

This is linear in the original predictors.

When we use a non-linear kernel (e.g. RBF), the model is no longer linear in the original feature space.

Instead, we implicitly transform the data:

$$
x \longrightarrow \phi(x)
$$

and optimise:

$$
w^\top \phi(x) + b
$$

This expression is still **linear in the transformed space**, but that transformed space may be much higher dimensional.

We do not need to specify what that space looks like.

Conceptually:

* In the original space → boundary may look curved.
* In the transformed space → the boundary is still a hyperplane.
* The margin maximisation principle is unchanged.

So nothing about the optimisation philosophy changes.

We are still:

* Maximising the margin.
* Penalising violations via $C$.
* Controlling geometric complexity.

Only the geometry of the space has changed.

### Intuition

You can think of it as:

> If data are not separable in 2D, we lift them into a richer space where separation becomes possible.

Then we apply the exact same maximum-margin logic.

We never explicitly compute that higher-dimensional space — the kernel function handles it implicitly.

But for our purposes, the key idea remains:

> Linear SVM = straight boundary in original space
> RBF SVM = curved boundary in original space
> Both maximise margin in some space
:::

In [ ]:
svm_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_svm_rbf = svm_rbf.predict(X_test_scaled)
y_score_svm_rbf = svm_rbf.decision_function(X_test_scaled)

#### Metrics

In [ ]:
print("RBF SVM metrics:")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_svm_rbf), 3))

print("F1 score:",
      round(f1_score(y_test, y_pred_svm_rbf), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_score_svm_rbf), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_svm_rbf))

#### PR curve (RBF)

In [ ]:
precision_rbf, recall_rbf, _ = precision_recall_curve(y_test, y_score_svm_rbf)

plt.figure(figsize=(6,5))
plt.plot(recall_rbf, precision_rbf)
plt.axhline(positive_rate, linestyle='--', color='black')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve (RBF SVM)")
plt.show()

#### Linear vs RBF interpretation

In this case:

* RBF ≈ Linear in balanced accuracy (both 0.5).
* Both produce identical confusion matrices.
* Neither improves classification beyond majority baseline.
* PR-AUC for both is slightly above positive-rate baseline, suggesting weak ranking information.

This suggests:

* Non-linearity does not materially improve classification.
* Signal-to-noise ratio is extremely low.
* Feature set may be insufficient.
* Thresholding at 0 may collapse ranking signal into majority prediction.

If neither beats majority baseline convincingly:

* Market efficiency may dominate.
* Features may not contain economically exploitable information.
* Signal may be economically tiny.

#### Conceptual reflection

Important:

* In finance, predictive improvements are often extremely small.
* Overfitting risk is extremely high.
* Out-of-sample robustness matters more than headline accuracy.
* Hard classification accuracy can hide weak ranking signal.
* Balanced accuracy is essential under drift and imbalance.

Beating:

* 0.5 balanced accuracy
* Positive-rate PR-AUC baseline

is the real test.

Here, the SVM models fail to beat the majority classifier in hard classification terms.

## 2.6 Decision Trees

Support Vector Machines construct separating hyperplanes.

Decision Trees take a completely different approach.

Instead of a global linear (or kernelised) boundary, trees:

> Partition the feature space into rectangular regions using a sequence of binary splits.

Where SVM constructs one global separating surface,
trees build a sequence of **local threshold rules**.

This makes their geometry fundamentally different from margin-based classifiers.


### 2.6.1 How a decision tree works

At each node, the algorithm:

1. Selects a feature $X_j$
2. Selects a threshold $t$
3. Splits the data into:

   * $X_j \le t$
   * $X_j > t$

The split is chosen to maximise **impurity reduction**.

In practice, this means:

> Among all possible features and thresholds, the tree selects the split that produces the largest decrease in class mixing.

---

### Visual illustration of a split

In [ ]:
# ----------------------------------------------------------
# 1. Simple synthetic 2D classification data
# ----------------------------------------------------------

X, y = make_blobs(
    n_samples=80,
    centers=2,
    cluster_std=1.5,
    random_state=3
)

# ----------------------------------------------------------
# 2. Choose a feature and threshold manually (like a tree would)
# ----------------------------------------------------------

feature_index = 0     # X1
threshold = np.median(X[:, feature_index])

# Split data
left = X[:, feature_index] <= threshold
right = X[:, feature_index] > threshold

# ----------------------------------------------------------
# 3. Plot
# ----------------------------------------------------------

plt.figure(figsize=(7,6))

# Plot points
plt.scatter(X[y==0,0], X[y==0,1], label="Class 0")
plt.scatter(X[y==1,0], X[y==1,1], label="Class 1")

# Draw vertical split
plt.axvline(threshold, linestyle="--", linewidth=3)

# Shade regions
plt.fill_betweenx(
    y=[X[:,1].min()-1, X[:,1].max()+1],
    x1=X[:,0].min()-1,
    x2=threshold,
    alpha=0.1
)

plt.fill_betweenx(
    y=[X[:,1].min()-1, X[:,1].max()+1],
    x1=threshold,
    x2=X[:,0].max()+1,
    alpha=0.05
)

plt.title("Decision Tree Split:  X₁ ≤ t  vs  X₁ > t")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()


#### Impurity measures (classification)

Two common criteria:

**Gini impurity**

$$
G = 1 - \sum_{k=1}^{K} p_k^2
$$

* Measures probability of misclassification if randomly labeling according to node distribution.
* Faster computationally.
* Default in `scikit-learn`.

**Entropy**

$$
H = -\sum_{k=1}^{K} p_k \log p_k
$$

* Measures information disorder.
* Penalises evenly mixed nodes slightly more strongly.

In practice:

* Gini and entropy often produce similar trees.
* Differences are usually minor.

Important:

> Both are greedy, local criteria.
> Trees do not globally optimise future splits.


### 2.6.2 Decision tree on stock market data

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree.fit(X_train_imputed, y_train)

y_pred_tree = tree.predict(X_test_imputed)
y_score_tree = tree.predict_proba(X_test_imputed)[:,1]

print("Decision Tree metrics:")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_tree), 3))

print("F1 score:",
      round(f1_score(y_test, y_pred_tree), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_score_tree), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_tree))

Observed:

* Balanced accuracy: **0.511**
* F1 score: **0.776**
* PR-AUC: **0.686**
* Confusion matrix:

$$
\begin{bmatrix}
8 & 49 \
15 & 111
\end{bmatrix}
$$

#### Interpretation using actual results

Let us interpret carefully relative to baselines.

**1. Balanced accuracy = 0.511**

Baseline balanced accuracy (random or majority) ≈ 0.5.

So:

* Improvement = +0.011.
* Extremely small.
* Statistically possibly indistinguishable from noise.

This suggests:

> The tree is extracting very little true signal.

**2. Confusion matrix**

$$
\begin{bmatrix}
8 & 49 \
15 & 111
\end{bmatrix}
$$

Interpretation:

* Only **8 true negatives**
* 49 false positives
* 15 false negatives
* 111 true positives

The tree is heavily biased toward predicting **Up**.

It behaves similarly to majority baseline, but:

* Slightly improves detection of Down months (8 correct vs 0 under majority).
* Still misclassifies most Down months.

This indicates:

> The model partially reacts to minority class but not strongly.

**3. F1 score = 0.776**

F1 appears high.

However:

* Positive class dominates.
* Predicting “Up” frequently inflates F1.
* Therefore F1 alone is misleading.

This is why balanced accuracy is more informative here.

**4. PR-AUC = 0.686**

Positive class prevalence ≈ 0.689 (from earlier random baseline).

PR-AUC = 0.686 is **slightly below prevalence**.

This means:

> The tree does not meaningfully improve ranking of Up vs Down months.

Even though F1 looks decent,
the ranking ability is not better than baseline.

**Overall conclusion for tree**

The tree:

* Does not meaningfully beat baseline in balanced accuracy.
* Does not improve PR-AUC.
* Mostly learns upward drift.

This is typical in equity prediction:

> Non-linearity alone does not create predictive power.

### 2.6.3 Visualising the tree (why trees are useful)

One key advantage of trees is interpretability.

In [ ]:
plt.figure(figsize=(18,8))
plot_tree(
    tree,
    feature_names=X_train.columns,
    class_names=["Down","Up"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree (max_depth=4)")
plt.show()

What this gives us:

* Explicit thresholds.
* Clear hierarchical rules.
* Direct mapping from economic variables to prediction.

Example rule (illustrative):

> If PE10_lag1 > 28.5
> and Inflation > 0.04
> → predict Down.

This level of transparency:

* Is impossible with RBF SVM.
* Is harder with Random Forest.
* Is central in policy or finance explainability discussions.


#### Explainability comparison: Trees vs SVM

| Model         | Transparency                 |
| ------------- | ---------------------------- |
| Decision Tree | Explicit if–then rules       |
| Random Forest | Aggregated, less transparent |
| SVM (linear)  | Coefficients interpretable   |
| SVM (RBF)     | Highly opaque                |

Trees are particularly attractive when:

* Interpretability matters.
* Economic reasoning is required.
* Model governance is important.

## 2.7 Feature selection

Feature selection is critical in financial data due to:

* Weak signals
* High noise
* Many correlated predictors

We distinguish:

### 2.7.1 Filter methods

Independent of model.

Examples:

* Mutual information
* Correlation screening
* Univariate F-tests


### 2.7.2 Recursive feature elimination (RFE)

RFE:

* Fits model.
* Removes weakest feature.
* Repeats recursively.

In [ ]:
rfe = RFE(estimator=tree, n_features_to_select=5)
rfe.fit(X_train_imputed, y_train)

selected_features = X_train.columns[rfe.support_]
print(selected_features)

RFE is:

* Model-based.
* Iterative.
* Useful when interactions matter.

### 2.7.3 Sequential feature selection (SFS / sometimes called SSE informally)

Sequential methods:

* Forward selection: start empty, add features.
* Backward elimination: start full, remove features.

In [ ]:
sfs = SequentialFeatureSelector(
    tree,
    n_features_to_select=5,
    direction="forward"
)

sfs.fit(X_train_imputed, y_train)

print(X_train.columns[sfs.get_support()])

These methods:

* Are computationally heavier.
* Capture interactions better than simple filters.
* Reduce overfitting risk when signal is weak.

### Important caution

Feature selection must be:

* Performed on training data only.
* Recomputed inside CV folds if doing cross-validation.

Otherwise:

> You leak information and overestimate performance.


## 2.7 Random Forests

A single decision tree has high variance.

Small changes in data → very different tree.

Random Forest reduces variance using two mechanisms:

1. **Bootstrap aggregation (bagging)**
   Each tree is trained on a bootstrap sample of the training data.

2. **Random feature selection at each split**
   At each node, only a random subset of features is considered.

This reduces correlation between trees and stabilises predictions.

Formally:

$$
\hat{f}*{RF}(x) = \frac{1}{B} \sum*{b=1}^{B} T_b(x)
$$

Where:

* $T_b$ = tree $b$
* $B$ = number of trees

### 2.7.1 Random Forest on stock data

In [ ]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

y_pred_rf = rf.predict(X_test_imputed)
y_score_rf = rf.predict_proba(X_test_imputed)[:,1]

print("Random Forest metrics:")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_rf), 3))

print("F1 score:",
      round(f1_score(y_test, y_pred_rf), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_score_rf), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))

#### Interpretation framework

Relative to:

* Random baseline
* Majority baseline
* Single decision tree
* Linear / RBF SVM

We ask:

1. Does RF improve balanced accuracy meaningfully?
2. Does PR-AUC exceed positive rate clearly?
3. Does minority recall improve?

#### What the actual metrics means

* **Balanced accuracy = 0.52**
  Slightly above 0.50 → small but real improvement over random/majority.

* Recall breakdown:

  * Down recall = 14 / 57 ≈ **0.25**
  * Up recall = 100 / 126 ≈ **0.79**
  * RF still favors “Up”, but not as extremely as SVM.

* **PR-AUC = 0.697**, above positive-rate baseline (~0.689).
  → Some ranking ability.

**Interpretation:**
Random Forest extracts **weak but non-zero signal**.
It improves minority detection slightly, but drift still dominates.


## 2.8 Balanced Random Forest

Financial data often exhibit:

* More Up months than Down months.
* Models collapsing to majority prediction.

Balanced Random Forest modifies the bootstrap procedure.

Instead of sampling bootstrap data uniformly:

> Each tree is trained on a **balanced subsample**.

This means:

* Equal number of Up and Down months per tree.
* Forces model to learn minority structure.

### 2.8.1 Fit Balanced RF

In [ ]:
brf = BalancedRandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

brf.fit(X_train_imputed, y_train)

y_pred_brf = brf.predict(X_test_imputed)
y_score_brf = brf.predict_proba(X_test_imputed)[:,1]

print("Balanced RF metrics:")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_brf), 3))

print("F1 score:",
      round(f1_score(y_test, y_pred_brf), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_score_brf), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_brf))

#### What to expect

Balanced RF often:

* Increases recall for Down months.
* Improves balanced accuracy.
* May reduce precision for Up months.

Trade-off:

Better symmetry
vs
More false positives

This is a **policy choice**:

* Do we care about detecting downturns?
* Or avoiding false alarms?

#### What changed in terms of metrics?

* Down recall = 20 / 57 ≈ **0.35**
* Up recall = 90 / 126 ≈ **0.71**

Balanced RF:

* Improves minority recall meaningfully.
* Sacrifices some majority recall.
* Increases balanced accuracy.

F1 drops slightly (because fewer Up predictions).

**Interpretation:**
Balanced RF better addresses class imbalance.
It trades some precision for better symmetry across classes.

This is the first model that clearly improves *balanced performance* rather than simply riding drift.

## 2.9 Gradient Boosting

Random Forest reduces variance.

Boosting reduces bias.

Instead of building trees independently:

Boosting builds trees sequentially:

$$
F_m(x) = F_{m-1}(x) + \eta T_m(x)
$$

Each tree corrects residual errors of previous ensemble.


### 2.9.1 Gradient Boosting on stock data

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    random_state=42
)

gb.fit(X_train_imputed, y_train)

y_pred_gb = gb.predict(X_test_imputed)
y_score_gb = gb.predict_proba(X_test_imputed)[:,1]

print("Gradient Boosting metrics:")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_pred_gb), 3))

print("F1 score:",
      round(f1_score(y_test, y_pred_gb), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_score_gb), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_gb))

#### Parameter intuition

* n_estimators → number of trees
* learning_rate → shrinkage (smaller = more stable)
* max_depth → tree complexity

High depth + many trees → overfitting.

Low learning rate + many trees → smoother approximation.

Boosting often performs very well in tabular datasets.

In finance:

* May capture subtle interactions.
* But still constrained by signal availability.

* Down recall ≈ 0.58
* Up recall ≈ 0.40

#### Interpretation of metrics

* Down recall ≈ 0.58
* Up recall ≈ 0.40

Boosting here:

* Overcorrected toward minority.
* Severely reduced Up recall.
* Dropped below 0.5 balanced accuracy.

This suggests:

* Overfitting to noise.
* Poor hyperparameter tuning.
* Or unstable signal in boosting framework.

**Conclusion:**
Boosting did not generalize well here.

:::callout-important

### What type of boosting is `GradientBoostingClassifier`?

`sklearn.ensemble.GradientBoostingClassifier` implements:

> Friedman-style Gradient Boosting using shallow CART trees.

It is:

* Sequential boosting
* Uses negative gradient of log-loss
* No histogram optimization
* No GPU acceleration

It is **not**:

* XGBoost
* LightGBM
* CatBoost

### Quick note on modern boosting libraries

#### XGBoost

* Regularized boosting
* Handles missing values natively
* Very strong baseline

**Example code**

```python
# --------------------------------------------
# XGBoost (classification)
# --------------------------------------------
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,      # L2 regularization
    reg_alpha=0.0,       # L1 regularization
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train_imputed, y_train)

y_pred_xgb = xgb.predict(X_test_imputed)
y_score_xgb = xgb.predict_proba(X_test_imputed)[:, 1]

```

XGBoost adds:

- Explicit regularization

- Missing value handling

- Optimized split finding

- Parallelization

#### LightGBM

* Histogram-based splits
* Extremely fast
* Leaf-wise growth (can overfit)

**Example code**

```python
# --------------------------------------------
# LightGBM
# --------------------------------------------
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(X_train_imputed, y_train)

y_pred_lgb = lgb_model.predict(X_test_imputed)
y_score_lgb = lgb_model.predict_proba(X_test_imputed)[:, 1]

```

*Key difference*:
LightGBM grows trees leaf-wise, not level-wise.
Faster, but more prone to overfitting if not tuned.

#### CatBoost

* Handles categorical features natively
* Ordered boosting to reduce leakage
* Very strong in tabular data

**Example code**

```python
# --------------------------------------------
# CatBoost
# --------------------------------------------
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    verbose=False,
    random_state=42
)

cat_model.fit(X_train_imputed, y_train)

y_pred_cat = cat_model.predict(X_test_imputed)
y_score_cat = cat_model.predict_proba(X_test_imputed)[:, 1]

```

Key strength:

- Native categorical handling

- Ordered boosting to reduce leakage

- Very strong in tabular finance problems

In practice:

> XGBoost / LightGBM usually outperform vanilla `sklearn` gradient boosting.

:::

## 2.10 Model comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Random",
        "Majority",
        "Linear SVM",
        "RBF SVM",
        "Decision Tree",
        "Random Forest",
        "Balanced RF",
        "Gradient Boosting"
    ],
    "Balanced accuracy": [
        0.5,
        balanced_accuracy_score(y_test, y_pred_majority),
        balanced_accuracy_score(y_test, y_pred_svm_linear),
        balanced_accuracy_score(y_test, y_pred_svm_rbf),
        balanced_accuracy_score(y_test, y_pred_tree),
        balanced_accuracy_score(y_test, y_pred_rf),
        balanced_accuracy_score(y_test, y_pred_brf),
        balanced_accuracy_score(y_test, y_pred_gb)
    ]
})

print(results)

Interpretation:

If all models cluster around 0.50–0.53:

> The data is weakly predictive.

If boosting clearly dominates:

> Non-linear interactions + bias reduction matter.

If Balanced RF improves minority detection significantly:

> Imbalance handling matters more than algorithmic complexity.

**What actually happened here**

1. Linear margin-based methods collapsed to majority behaviour.
2. Single tree extracts tiny signal.
3. Ensemble trees improve modestly.
4. Balanced RF performs best.
5. Boosting underperforms.

*Key lesson:*

> Non-linearity + imbalance-aware sampling helps slightly —
> but signal remains economically tiny.

Even 0.533 balanced accuracy is only modestly above random.

In equity prediction, that is actually typical.


## 2.11 Time-aware cross-validation (example with Balanced RF)

In [ ]:
X_train_df = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

y_train_series = pd.Series(y_train, index=X_train.index)

tscv = TimeSeriesSplit(n_splits=5)

bal_acc_scores = []
f1_scores = []
pr_auc_scores = []

for train_idx, val_idx in tscv.split(X_train_df):

    X_tr = X_train_df.iloc[train_idx]
    X_val = X_train_df.iloc[val_idx]

    y_tr = y_train_series.iloc[train_idx]
    y_val = y_train_series.iloc[val_idx]

    model = BalancedRandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_val)
    y_score = model.predict_proba(X_val)[:,1]

    bal_acc_scores.append(
        balanced_accuracy_score(y_val, y_pred)
    )

    f1_scores.append(
        f1_score(y_val, y_pred)
    )

    pr_auc_scores.append(
        average_precision_score(y_val, y_score)
    )

print("Time-aware CV results (mean across folds):")
print("Balanced accuracy:", round(np.mean(bal_acc_scores), 3))
print("F1 score:", round(np.mean(f1_scores), 3))
print("PR-AUC:", round(np.mean(pr_auc_scores), 3))

Important:

* No shuffling.
* Forward expanding window.
* Prevents look-ahead bias.
* Each fold validates on strictly future observations.


### Interpretation of cross-validation

* Cross-validated balanced accuracy ≈ 0.507
* Cross-validated PR-AUC close to positive rate
* F1 modest but unstable across folds

This indicates:

* Signal is weak.
* Performance varies across regimes.
* The model does not consistently outperform randomness across time.

Time-aware CV simulates:

* Re-training the model repeatedly
* Moving forward through time
* Validating only on unseen future data
* Experiencing different macro environments

In finance:

> Stability across time is more important than one good split.

### Finalised Balanced Random Forest Model

After cross-validation, we now train the **final model** on the full training period.

In [ ]:
final_brf = BalancedRandomForestClassifier(
    n_estimators=200,
    random_state=42
)

final_brf.fit(X_train_df, y_train_series)

y_test_pred = final_brf.predict(X_test_imputed)
y_test_score = final_brf.predict_proba(X_test_imputed)[:,1]

print("Final Balanced RF (Test Set Metrics):")

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, y_test_pred), 3))

print("F1 score:",
      round(f1_score(y_test, y_test_pred), 3))

print("PR-AUC:",
      round(average_precision_score(y_test, y_test_score), 3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

### Final model interpretation (relative to CV)

Suppose results resemble:

* Test balanced accuracy ≈ 0.533
* CV balanced accuracy ≈ 0.507

This gap tells us:

* The fixed test split was relatively favourable.
* Performance is sensitive to sample window.
* Gains are not fully stable through time.

If CV ≈ Test:

* Model generalises consistently.
* Signal more credible.

If CV << Test:

* Risk of overfitting to particular regime.
* Gains may not persist.

In financial modelling:

> Cross-validation stability matters more than the absolute metric value.

---

## Why we report all three metrics

Balanced accuracy:

* Measures symmetric class performance.
* Immune to majority dominance.

F1 score:

* Reflects precision–recall tradeoff.
* Useful when minority class detection matters.

PR-AUC:

* Baseline equals positive class prevalence.
* Directly measures signal extraction under imbalance.

A credible financial model should:

* Beat 0.5 balanced accuracy consistently
* Beat positive-rate PR-AUC baseline
* Show stability across time folds

---

# Conceptual takeaway

Time-aware cross-validation transforms evaluation from:

> “Did it work once?”

to

> “Does it work repeatedly across market regimes?”

In equity prediction:

* Noise is high.
* Regimes change.
* Small edges disappear quickly.

A model with:

* Balanced accuracy ≈ 0.53 on one split
* But ≈ 0.50 across time folds

is not a strong trading signal.

It is a weak statistical pattern at best.
This is exactly what we expect in equity return prediction.

## 2.12 Trees and SVM in regression (important clarification)

We have so far focused on **classification**.

It is crucial to understand that:

> These methods are not classification-only algorithms.

They generalise naturally to regression.

### For trees

In classification trees, splits are chosen to reduce class impurity:

* Gini impurity
* Entropy

In regression trees, we replace impurity with variance reduction.

The splitting criterion becomes:

$$
\text{MSE} = \frac{1}{n} \sum (y_i - \bar{y})^2
$$

So instead of reducing class mixing, the tree now:

> Reduces outcome variance within each region.

In classification:

* Terminal nodes output a class label (majority vote).

In regression:

* Terminal nodes output the **mean** of observations in that region.

The structure of the algorithm remains identical:

* Recursive binary splits
* Axis-aligned partitions
* Ensembles reduce variance (Random Forest)
* Boosting reduces bias

The only change is the loss criterion.

Models:

* `DecisionTreeRegressor`
* `RandomForestRegressor`
* `GradientBoostingRegressor`

So conceptually:

> Trees partition space the same way — only the target and loss change.

### For SVM

In classification, we defined the decision function:

$$
f(x) = w^\top x + b
$$

The classification rule depends on the **sign** of $f(x)$.

We then introduced the idea that SVM:

* Maximises the margin
* Penalises violations
* Controls the trade-off via $C$

In regression, we keep the same geometric philosophy:

> Find a function that is as flat as possible, while tolerating small errors.

Instead of separating two classes, we now approximate a continuous target.

#### Regression SVM (SVR)

The optimisation becomes:

$$
\min \frac{1}{2}||w||^2 + C \sum \max(0, |y_i - \hat{y}_i| - \epsilon)
$$

Where:

* $||w||^2$ controls flatness (smoothness of the function)
* $C$ controls penalty strength
* $\epsilon$ defines a “no-penalty” tube around the prediction

### Geometric intuition

Classification SVM:

* Maximises distance between classes.

Regression SVM:

* Maximises flatness of the function
* Allows small deviations inside an $\epsilon$-tube
* Penalises only meaningful deviations

So instead of margins between classes, we now have:

> A margin around the regression function.

Inside that tube:

* Errors are ignored.

Outside that tube:

* Errors are penalised linearly.

Another way to see it is:

> SVR finds the flattest possible function that keeps most data points within an ε-tolerance band. Points outside that band are penalised.

Or more intuitively:

- Step 1: Make the function as simple/flat as possible.
- Step 2: Only bend it if necessary to keep large errors under control.
- Step 3: Ignore small noise entirely.

#### Comparison to OLS

Ordinary Least Squares:

$$
\min \sum (y_i - \hat{y}_i)^2
$$

* Penalises all errors
* Penalises large errors quadratically

SVR:

* Ignores small errors
* Penalises large errors linearly
* Encourages flat (low-variance) functions

This can be appealing in finance, where:

* Returns are noisy
* Tiny errors are economically irrelevant
* We care about meaningful deviations

#### Implementation

```python
from sklearn.svm import SVR

svr = SVR(kernel="rbf", C=1.0, epsilon=0.1)
svr.fit(X_train_scaled, y_train_regression)
```

Conceptually:

Classification SVM
→ maximise separation between classes

Regression SVM
→ maximise flatness while tolerating small noise

Both:

> Control geometry through regularisation.

No new philosophy — only a new target.

## 2.13 Feature importance vs SHAP

Tree-based models naturally provide **feature importance scores**.

But we must be precise about what these mean.

### 2.13.1 Random Forest feature importance

In scikit-learn:

In [ ]:
importances = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

plt.figure(figsize=(8,6))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance (Impurity-based)")
plt.xlabel("Mean decrease in impurity")
plt.show()

print(importances)

Our plot shows:

Top features:

1. Return_lag1
2. Return_lag3
3. Earnings_Yield_lag1
4. Inflation
5. MA3 / MA12

Interpretation:

* Short-term momentum dominates.
* Valuation matters but less than recent returns.
* Macro variables contribute but do not dominate.

Important:

> The model relies heavily on recent returns —
> consistent with weak momentum effect.

However:

* Importance magnitudes are similar.
* No single dominant driver.
* Signal likely diffuse and weak.


#### What does this measure?

For each feature:

* How often it is used in splits
* How much impurity it reduces
* Averaged across all trees

Formally:

It measures total **Gini impurity reduction** attributed to that feature.

#### Important limitations

1. It does **not** show direction (positive vs negative effect).
2. It does **not** show whether effect is linear or threshold-based.
3. It can be biased toward:

   * Continuous variables
   * High-cardinality variables
4. It does **not imply causality**.

It answers:

> “Which variables are used most often to reduce classification error?”

It does *not* answer:

> “Does high PE10 increase or decrease crash probability?”


### 2.13.2 SHAP values

Now we compute SHAP for the same Random Forest.

In [ ]:
X_test_df = pd.DataFrame(
    X_test_imputed,
    columns=X_test.columns,
    index=X_test.index
)
explainer = shap.TreeExplainer(rf)

shap_values = explainer(X_test_df)

shap.summary_plot(
    shap_values[:, :, 1],   # class "Up"
    X_test_df
)

#### SHAP interpretation

The SHAP summary plot explains how each feature contributes to the model’s predictions.

Each dot represents:

* One observation
* For one feature
* With that feature’s contribution to the model’s prediction

**How to read the plot (mechanically)**

For each feature row:

1. **Horizontal position (SHAP value)**

   * Right of zero → pushes prediction toward **Up**
   * Left of zero → pushes prediction toward **Down**

2. **Colour (feature value)**

   * Red → high feature value
   * Blue → low feature value

3. **Spread (magnitude)**

   * How far dots extend from zero indicates how strongly the feature influences predictions.

**What does a SHAP value represent?**

For classification models, SHAP explains the model output **before thresholding**.

It decomposes each prediction:

$$
f(x) = \phi_0 + \sum_j \phi_j
$$

Where:

* $\phi_0$ = baseline prediction (average model output)
* $\phi_j$ = contribution of feature $j$

So SHAP values show:

> How much each feature moves the prediction away from the average prediction.

If the baseline probability of “Up” is 0.60:

* Positive SHAP → increases probability above 0.60
* Negative SHAP → decreases probability below 0.60

It does **not** mean:

> The feature causes the market to go Up.

It means:

> The model associates that feature value with higher or lower Up probability.

**Interpreting our specific plot**

Below is the economic interpretation of what we see in our actual figure.

*1. `Return_lag3` (3-month momentum)*

Look at the row for **Return_lag3**:

* Most dots lie slightly on the **negative** side of zero.
* The strongest positive SHAP values tend to correspond to **high Return_lag3** (red points on the right).

This suggests:

* High recent 3-month returns sometimes push predictions toward **Up**.
* But many observations contribute slightly toward **Down**.
* The effect is not strong or consistent.

Magnitude:

* SHAP values mostly lie between roughly −0.03 and +0.03.
* This is a **small shift** in predicted probability.

Interpretation:

> The model extracts weak and unstable momentum effects.

This is exactly what we expect in noisy financial data.

*2. `MA12` and `Return_lag1` (short-term momentum)*

For MA12 and Return_lag1:

* High values (red) tend to appear on the **positive SHAP side**.
* Low values (blue) tend to appear more often on the **negative side**.

This is a cleaner directional pattern.

Interpretation:

> When recent momentum is strong, the model slightly increases the probability of Up.
> When recent momentum is weak, it slightly decreases it.

Economically plausible — but again:

Magnitude remains small.

*3. Valuation variables (`Earnings_Yield_lag1`, `PE10_lag1`)*

Pattern:

* High earnings yield (cheap market) → mildly positive SHAP.
* Low earnings yield (expensive market) → mildly negative SHAP.

Interpretation:

> The model learns a mild valuation effect consistent with mean reversion.

But:

* Effects are small.
* Contributions rarely exceed ±0.04.

This means valuation plays a role — but not a dominant one.

*4. Macro variables(`Inflation`, `Interest_Rate_lag1`, `Interest_Rate_change`)*

Observed pattern:

* Mixed red and blue on both sides.
* No strong monotonic direction.
* Narrow SHAP range.

Interpretation:

> Macro effects appear regime-dependent and unstable.
> The model struggles to extract consistent directional signal from macro variables.

**Crucial observation: magnitude**

Across all features:

* Most SHAP values lie within approximately ±0.05.
* No feature produces large, dominant contributions.

This tells us:

* The model’s predictions are **marginal adjustments**.
* No single variable drives decisions.
* Signal-to-noise ratio is low.

This aligns with:

* Balanced accuracy ≈ 0.50–0.53
* PR-AUC only slightly above baseline
* Typical behaviour in equity return prediction

**What SHAP tells us about credibility**

SHAP confirms:

* The model is not relying on pathological shortcuts.
* Momentum and valuation play modest roles.
* Effects are economically interpretable.
* No extreme or implausible rules are driving predictions.

In short:

> The model extracts small, plausible signals — not dramatic deterministic rules.

This is realistic for financial markets.

#### RF Feature Importance vs SHAP

Random Forest impurity-based importance tells us:

> Which variables the model splits on most.

SHAP tells us:

> How each variable pushes predictions, in which direction, and by how much.

| Property               | RF Importance | SHAP |
| ---------------------- | ------------- | ---- |
| Shows magnitude        | Yes           | Yes  |
| Shows direction        | No            | Yes  |
| Observation-specific   | No            | Yes  |
| Captures heterogeneity | No            | Yes  |
| Additive decomposition | No            | Yes  |

SHAP is therefore strictly more informative for interpretation.


### Quick note: predicting Up vs Down

In this application:

* The market goes Up more often than Down.
* Predicting “Up” frequently already gives reasonable F1.

So the more economically valuable signal is:

> Does the model correctly identify Down months?

If false negatives (missed Down markets) are costly,
then recall on Down months matters more than overall F1.

:::callout-note
In this application (predicting whether the market goes **Up** next month):

* **False Positive (FP)** = model predicts *Up*, market actually goes *Down*
* **False Negative (FN)** = model predicts *Down*, market actually goes *Up*

### Which matters more?

In most investment contexts:

> **False Positives are typically more costly.**

Why?

* A False Positive means you stay invested (or go long) before a Down month.
* That exposes you to losses.
* A False Negative simply means you miss a positive month — an opportunity cost, but not an actual loss.

So economically:

* **FP → potential capital loss**
* **FN → missed upside**

Therefore, in many portfolio applications:

> Avoiding False Positives (i.e., correctly detecting Down months) is more important.

That said, the answer depends on strategy:

* Long-only investor → FP more harmful
* Market-timing hedge strategy → both matter
* Short-selling strategy → FN could matter more

But under a typical long-only allocation framework:

> **Correctly identifying Down markets (reducing FP) is usually the priority.**

:::

In practice:

* Balanced accuracy is more informative than raw accuracy.
* PR-AUC helps evaluate positive-class ranking performance.

The SHAP plot shows that:

> No feature strongly pushes predictions toward Down.
> The model is cautious and produces small probability adjustments.

This reinforces the conclusion:

> The constraint is weak economic signal — not algorithmic sophistication.


# 🎯 Key Takeaways

**Technical**:
1. Cross-validation gives robust performance estimates
2. GridSearchCV for hyperparameter tuning
3. Fairness impossibility when base rates differ
4. SVM: Maximum margin + kernel trick
5. Trees: Interpretable but overfit
6. Random Forest: Ensemble reduces variance

**Practical**:
1. Fairness is a values question
2. Always establish baselines
3. Time series need temporal splits
4. High accuracy ≠ useful model
5. Healthy skepticism about ML hype



# 📚 Next Week

**Dimensionality Reduction & Unsupervised Learning**
- Too many features?
- No labels?
- Finding structure without supervision
